# Этап 4. Часть II. Вторая версия анализа (проект V2)

**Тема:** «Разработка интеллектуального Telegram-бота на основе языковой модели для консультирования по нормативной документации в области электроэнергетики (ПУЭ, ПТЭЭП, ГОСТ)».

**Репозиторий:** https://github.com/kirag-ozyaz/Diplom_AI

Ноутбук второй части отчёта по этапу 4: разделы 9-17, номера таблиц (17-32) и рисунков (12-24) совпадают с отчётом Word. Часть II описывает версию V2: корпус с вручную выверенными главами 1.1-1.9, расширенный набор вопросов с эталоном по дословному тексту ПУЭ, пять прогонов поиска, генерацию ответов и их оценку. Все числа считаются из сохранённых файлов результатов; Milvus, Ollama и видеокарта не нужны, достаточно выполнить все ячейки (Run All).

**Где ноутбук берёт данные.** Сначала папка `data_V2` рядом с ноутбуком (в ней сохранена структура папок репозитория), затем корень репозитория Diplom_AI (первая папка вверх от текущей, где есть `V2/results/retrieval`). Ноутбук читает:

- файлы метрик `metrics.json` прогонов поиска в `V2/results/retrieval/split` и `split_extra`; опыт определяется по настройкам внутри файла, поэтому имена папок прогонов не важны (если прогонов одного опыта несколько, берётся последний);
- набор вопросов `V2/data/rag_eval_questions/eval_questions_split.json` и корпус `V2/data/chunked_merged_manual`;
- ответы модели и оценки судьи: `V2/results/generation/answers/*_rerank50/`, `judge/*_rerank50/`, для вопросов без ответа в ПУЭ - `answers/*_refusal/`, `judge/*_refusal/`;
- для сравнения с первой версией - файлы V1 `scripts/stage4_eval/results/retrieval/stage4_rank_results_{base,e5,rerank}.json` и `stage4_hitk_runs.jsonl`.

Повторный поиск (`RUN_LIVE`) и пересчёт метрик из сохранённой выдачи (`RUN_EVAL`) выключены (раздел 15). Первая часть отчёта (разделы 1-8) - ноутбук `Etap4_V1.ipynb`.

## 9. Задачи второй версии

Задачи второй версии вытекают из результатов первой (раздел 7). Общая задача остаётся прежней - ответ по ПУЭ (7-е издание) со ссылкой на пункт, таблицу или рисунок по схеме RAG. Поскольку без найденного фрагмента правильный ответ невозможен, во второй версии по-прежнему измеряется прежде всего качество поиска: попадает ли фрагмент с эталонным текстом в первые k результатов.

### 9.1. Причины перехода от версии V1 к версии V2

Анализ работы версии V1 показал, что текущие показатели (Hit@1 = 58,2-60,1 %, Hit@5 = 81,3-83,6 %) и качество генерации ответов не обеспечивают требуемой надёжности при работе с нормативной документацией ПУЭ. Основными причинами стали ошибки автоматического чанкинга, отсутствие ручной выверки исходных текстов и недостатки схемы зачёта по пункту. Часть чисел V1 нельзя считать надёжными, причём дело не в моделях, а в корпусе, эталоне и способе оценки. Исправлять это в старом коде значило бы менять уже принятые результаты, поэтому V2 сделан отдельным проектом в папке V2. Старые скрипты, корпус `data/chunked`, вопросы `eval_questions.json` и результаты первой версии сохранены без изменений и служат базой для сравнения.

В версии V2 решались следующие вопросы:

1. Переработка базы, обновление алгоритма парсинга и нарезки.
2. Переработка методологии оценки: переход на атомарные факты и составные вопросы.
3. Оценка влияния служебной разметки модели E5 (`query:` и `passage:`) на качество поиска.
4. Оценка прироста от переранжирования кросс-энкодером и его зависимости от числа кандидатов.
5. Проверка пользы поиска по изображениям (CLIP) в дополнение к текстовому поиску.
6. Оценка ответов модели по найденным фрагментам LLM-судьёй.
7. Проверка правильного отказа системы, если ответа в ПУЭ нет.

## 0. Подготовка: поиск данных и вспомогательные функции

In [1]:
# Флаги повторного расчёта (раздел 15). False - только чтение сохранённых результатов.
RUN_LIVE = False   # True - заново выполнить поиск по готовым коллекциям Milvus (нужны Milvus и видеокарта)
RUN_EVAL = False   # True - пересчитать metrics.json из сохранённой выдачи retrieval.json без Milvus

import json, re, statistics as st
from collections import Counter
from pathlib import Path
import subprocess, sys, urllib.request, zipfile
try:
    import plotly
except ImportError:   # другой компьютер или Colab без plotly
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)

# Если папки DATA рядом с ноутбуком нет (например, в Google Colab), она скачивается архивом с Google Drive.
DATA_URL = "https://drive.google.com/uc?export=download&id=1atHoDUS384LwU2R6TTHeA4CL16oqFXz8"
if not (Path.cwd() / "data_V2").is_dir() and not any((p / "V2" / "results" / "retrieval").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    print("Папка data_V2 не найдена, скачиваю архив...")
    urllib.request.urlretrieve(DATA_URL, "data_V2.zip")
    zipfile.ZipFile("data_V2.zip").extractall(Path.cwd())
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

def find_root(start=None):
    """Папка с данными: сначала data_V2 рядом с ноутбуком, затем корень репозитория Diplom_AI
    (первая папка вверх от текущей, где есть V2/results/retrieval)."""
    start = Path(start or Path.cwd()).resolve()
    if (start / "data_V2" / "V2" / "results" / "retrieval").is_dir():
        return start / "data_V2"
    for p in [start, *start.parents]:
        if (p / "V2" / "results" / "retrieval").is_dir():
            return p
    raise FileNotFoundError("Не найдены данные: положите папку data_V2 рядом с ноутбуком "
                            "или откройте ноутбук внутри репозитория Diplom_AI.")

ROOT = find_root()
V2RES = ROOT / "V2" / "results" / "retrieval"
V1RES = ROOT / "scripts" / "stage4_eval" / "results" / "retrieval"
GEN = ROOT / "V2" / "results" / "generation"
QFILE = ROOT / "V2" / "data" / "rag_eval_questions" / "eval_questions_split.json"
print("Данные:", ROOT.relative_to(Path.cwd()).as_posix() if ROOT.is_relative_to(Path.cwd()) else ROOT)

def num(v, d=1):
    """Число в русском формате: 81.3 -> '81,3'."""
    return "-" if v is None else f"{v:.{d}f}".replace(".", ",")

def pct(x, d=1):
    """Доля 0..1 в процентах с десятичной запятой."""
    return num(x * 100, d)

def md_table(header, rows):
    out = ["| " + " | ".join(map(str, header)) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(str(x) for x in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))

def caption(text):
    """Подпись таблицы или рисунка, как в отчёте Word."""
    display(Markdown(f"**{text}**"))

def _show(fig, title, ylabel, yrange=None, height=460):
    fig.update_layout(title=title, template="plotly_white", height=height,
                      margin=dict(t=88, b=48, l=56, r=24),
                      legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
    fig.update_yaxes(title=ylabel, **({"range": yrange} if yrange else {}))
    fig.show()

def bar(title, cats, series, ytitle="%", yrange=None, d=1):
    """Сгруппированная столбчатая диаграмма: series = [(подпись, [значения], цвет)]."""
    fig = go.Figure()
    for name, vals, color in series:
        fig.add_bar(name=name, x=[str(c) for c in cats], y=vals, marker_color=color,
                    text=[num(v, d) for v in vals], textposition="outside", cliponaxis=False,
                    hovertemplate="%{x}<br>" + name + ": %{text}<extra></extra>")
    fig.update_layout(barmode="group")
    _show(fig, title, ytitle, yrange)

EXP = ["o1", "o2", "o3", "o4", "o5"]
NAME = {"o1": "Опыт 1. Без префиксов", "o2": "Опыт 2. Префиксы E5", "o3": "Опыт 3. Реранкер, 20",
        "o4": "Опыт 4. Реранкер, 50", "o5": "Опыт 5. Реранкер, 50 + CLIP", "ctl": "Контроль (повтор опыта 2)"}
SHORT = {"o1": "Опыт 1", "o2": "Опыт 2", "o3": "Опыт 3", "o4": "Опыт 4", "o5": "Опыт 5", "ctl": "Контроль"}
COLOR = {"o1": "#9aa5b1", "o2": "#5b8def", "o3": "#f2a541", "o4": "#e4572e", "o5": "#2a9d8f", "ctl": "#7b61ff"}

def classify(m):
    """Опыт по настройкам внутри metrics.json."""
    s = (m.get("search_settings") or {}).get("mode") or {}
    if m.get("clip_enabled"): return "o5"
    if s.get("rerank_model"): return "o4" if s.get("candidate_depth", 0) >= 50 else "o3"
    if s.get("query_prefix", "") == "": return "o1"
    return "ctl" if s.get("collection", "").endswith("_e5") else "o2"

RUNS = {}
for f in sorted(V2RES.glob("split*/**/metrics.json")):
    m = json.loads(f.read_text(encoding="utf-8"))
    if m.get("qrels_version") != "split-720-exact-text-v1" or len(m.get("questions", [])) != 720:
        continue
    e = classify(m)
    if e not in RUNS or m["created_at"] > RUNS[e][1]["created_at"]:
        RUNS[e] = (f, m)
missing = [NAME[e] for e in EXP if e not in RUNS]
assert not missing, f"Нет прогонов: {missing}"
md_table(["Опыт", "Коллекция", "Кандидатов", "CLIP", "Файл метрик"],
         [[NAME[e], RUNS[e][1]["search_settings"]["mode"]["collection"],
           (RUNS[e][1]["search_settings"]["clip"]["candidate_depth"] if RUNS[e][1].get("clip_enabled")
            else RUNS[e][1]["search_settings"]["mode"].get("candidate_depth", "-")),
           "да" if RUNS[e][1].get("clip_enabled") else "нет",
           f"`{RUNS[e][0].relative_to(ROOT).as_posix()}`"] for e in [*EXP, "ctl"] if e in RUNS])

D = {e: {r["qid"]: r for r in RUNS[e][1]["questions"]} for e in RUNS}
QS = {q["id"]: q for q in json.loads(QFILE.read_text(encoding="utf-8"))["questions"]}
GROUPS = {"orig": lambda q: "." not in q, "parts": lambda q: "." in q, "all": lambda q: True}

def agg(e, group, k, level="content", key="hit"):
    """Среднее по группе вопросов: level = content (дословный абзац), clause (пункт) или facts."""
    rows = [r for q, r in D[e].items() if GROUPS[group](q)]
    if level == "facts":
        return sum(1 for r in rows if r["metrics"][k]["facts"][key]) / len(rows)
    return sum(r["metrics"][k][level][key] or 0 for r in rows) / len(rows)

Данные: data_V2


| Опыт | Коллекция | Кандидатов | CLIP | Файл метрик |
|---|---|---|---|---|
| Опыт 1. Без префиксов | diplom_v2_noprefix | - | нет | `V2/results/retrieval/split_extra/20261111_000201_350614/metrics.json` |
| Опыт 2. Префиксы E5 | diplom_v2_base | - | нет | `V2/results/retrieval/split/20261110_210122_991218/metrics.json` |
| Опыт 3. Реранкер, 20 | diplom_v2_rerank | 20 | нет | `V2/results/retrieval/split/20261110_224306_716179/metrics.json` |
| Опыт 4. Реранкер, 50 | diplom_v2_rerank | 50 | нет | `V2/results/retrieval/split_extra/20261111_000608_110945/metrics.json` |
| Опыт 5. Реранкер, 50 + CLIP | diplom_v2_rerank | 50 | да | `V2/results/retrieval/split/20261110_231109_145164/metrics.json` |
| Контроль (повтор опыта 2) | diplom_v2_e5 | - | нет | `V2/results/retrieval/split/20261110_221857_146890/metrics.json` |

## 10. База данных

**Корпус.** Источник - тексты ПУЭ из `data/raw` (только чтение, официальный текст не правится). Документы извлечены в Markdown с таблицами, формулами и изображениями и нарезаны на фрагменты (чанки). Главы 1.1-1.9 были исправлены вручную после извлечения и импортированы отдельным импортёром целиком, остальные главы взяты из прежней нарезки. Итоговый корпус - `V2/data/chunked_merged_manual` (таблица 17).

In [2]:
qs = list(QS.values())
orig = [q for q in qs if q["is_original_question"]]
sec = Counter(q["expected_clause"].split(".")[0] for q in orig)
chunks = ROOT / "V2" / "data" / "chunked_merged_manual"
cfiles = sorted(chunks.glob("*.chunked.jsonl")) if chunks.is_dir() else []
caption("Таблица 17 - Корпус второй версии")
if cfiles:
    recs = [json.loads(l) for f in cfiles for l in f.read_text(encoding="utf-8").splitlines() if l.strip()]
    md_table(["Показатель", "Значение"], [
        ["Документов (файлов чанков)", len(cfiles)], ["Чанков", len(recs)],
        ["С номером пункта (Clause)", sum(1 for r in recs if r["metadata"].get("Clause"))],
        ["С таблицами", sum(1 for r in recs if r["metadata"].get("contains_tables"))],
        ["Со ссылками на изображения", sum(1 for r in recs if r["metadata"].get("contains_images"))]])
else:
    print("Корпус не найден:", chunks)
print("Изображений в коллекции CLIP (diplom_v2_clip) - 390, по журналу загрузки load-clip; файлы изображений в пакет не входят.")
print("Для сравнения: в V1 корпус data/chunked содержал 2912 чанков.")

**Таблица 17 - Корпус второй версии**

| Показатель | Значение |
|---|---|
| Документов (файлов чанков) | 46 |
| Чанков | 3146 |
| С номером пункта (Clause) | 2987 |
| С таблицами | 144 |
| Со ссылками на изображения | 197 |

Изображений в коллекции CLIP (diplom_v2_clip) - 390, по журналу загрузки load-clip; файлы изображений в пакет не входят.
Для сравнения: в V1 корпус data/chunked содержал 2912 чанков.


**Вопросы.** Исходный набор - 268 вопросов по всем семи разделам ПУЭ (файл `eval_questions.json`, не изменялся). Многие исходные вопросы составные: в одном вопросе несколько требований из разных пунктов. Поэтому каждый вопрос вручную разбит на части X.1, X.2 … - одно требование одного пункта. Исходные вопросы сохранены дословно и с прежними номерами, части добавлены рядом. Итоговый набор вопросов (файл `eval_questions_split.json`, версия split-720-v1): 720 вопросов = 268 исходных + 452 части (таблица 18).

In [3]:
caption("Таблица 18 - Исходные вопросы по разделам ПУЭ")
md_table(["Раздел ПУЭ", *sorted(sec), "Итого"], [["Вопросов", *[sec[s] for s in sorted(sec)], sum(sec.values())]])
print(f"Всего вопросов: {len(qs)} = {len(orig)} исходных + {len(qs) - len(orig)} частей")

**Таблица 18 - Исходные вопросы по разделам ПУЭ**

| Раздел ПУЭ | 1 | 2 | 3 | 4 | 5 | 6 | 7 | Итого |
|---|---|---|---|---|---|---|---|---|
| Вопросов | 42 | 45 | 35 | 41 | 39 | 31 | 35 | 268 |

Всего вопросов: 720 = 268 исходных + 452 частей


**Эталон** (qrels - таблица соответствия «вопрос - обязательные абзацы ПУЭ»). Эталонный ответ каждого вопроса - дословные абзацы корпуса, каждый абзац - отдельный обязательный факт. Всего 1733 факта, все найдены в чанках дословно (0 без чанка), проверка файла пройдена. Чанк, содержащий эталонный абзац, получает оценку 2, чанк с нужным пунктом без эталонного текста - оценку 1. В V1 у каждого вопроса был один эталонный пункт, и попаданием считался чанк этого пункта. В V2 засчитывается только чанк, содержащий дословный эталонный абзац, а поиск номера пункта в тексте в зачёте не участвует. Для сравнения с первым проектом в разделе 13.6 у V2 используется зачёт по пункту, как в V1.

## 11. Параметризация данных

**Сегментация (чанкинг).** Парсер V2 (`V2/scripts/prepare_chunks.py` / `src/retrieval_eval/preparation.py`) разбирает Markdown по уровням: документ (Document), раздел (Section), глава (Chapter), параграф (Paragraph) и пункт (Clause). Правило: один пункт ПУЭ - один чанк. Ячейка ниже выводит из корпуса пример чанка (пункт 1.1.39).

In [4]:
ex = next((r for r in recs if r["metadata"].get("Clause") == "1.1.39"), None) if cfiles else None
if ex:
    show = {"id": ex["id"], "metadata": {k: ex["metadata"].get(k) for k in
            ("Document", "Section", "Chapter", "Paragraph", "Clause", "contains_tables", "contains_images",
             "source_file", "chunk_type", "passage_prefix")}, "content": ex["content"]}
    print(json.dumps(show, ensure_ascii=False, indent=2))

{
  "id": "v2_ed4b103a137bc6598cdee11b5faad547df8ee1b4",
  "metadata": {
    "Document": "data/raw/Нормативная база/ПУЭ/DOCX/1.1.docx",
    "Section": "",
    "Chapter": "1.1",
    "Paragraph": "",
    "Clause": "1.1.39",
    "contains_tables": false,
    "contains_images": false,
    "source_file": "data/raw/Нормативная база/ПУЭ/DOCX/1.1.docx",
    "chunk_type": null,
    "passage_prefix": null
  },
  "content": "1.1.39. Вновь сооруженные и реконструированные электроустановки вводятся в промышленную эксплуатацию только после их приемки согласно действующим положениям.\r\n\r\nТекст документа сверен по:\r\n\r\nнормативно-производственное издание\r\n\r\nМ.: Издательство НЦ ЭНАС, 2002"
}


Основные отличия V2 от V1:

- **Префиксы модели эмбеддингов** (`passage:` / `query:`): в версии V2 префикс `passage:` добавляется к тексту фрагмента при индексации E5 (`intfloat/multilingual-e5-base`), а `query:` - к вопросу при поиске во всех опытах, кроме опыта 1; в версии V1 префиксы применялись только в опытах 2 и 3.
- **Выверенность иерархии в metadata:** заполнены ранее пустые поля иерархии (например, Chapter: "Глава 1.1 Общие требования"), что позволяет формировать точный полный заголовок нормативного документа.
- **Привязка к дословным абзацам (Exact Text Qrels):** чанки V2 строго приведены в соответствие с 1733 дословными абзацами эталонных ответов (файл эталона `eval_qrels_split.json`), а в метаданные добавлен тип разбиения (`chunk_type`).

**Общие параметры всех прогонов.** Milvus (http://localhost:19530), индекс AUTOINDEX, мера COSINE; текстовая модель `intfloat/multilingual-e5-base` (размерность 768, нормированные векторы); устройство cuda (NVIDIA RTX 5060 Ti, 16 ГБ); пакет загрузки 16; глубина выдачи 20; k = 1, 3, 5, 10, 20 (таблица 19). Таблица 19 строится по настройкам внутри `metrics.json`.

In [5]:
def params(e):
    m = RUNS[e][1]; s = m["search_settings"]["mode"]
    c = m["search_settings"].get("clip") if m.get("clip_enabled") else None
    pref = "нет" if not s.get("query_prefix") else f'{s["query_prefix"].strip()} / {s.get("passage_prefix", "").strip()}'
    return [NAME[e].replace("Опыт ", ""), s["collection"] + (f' + {c["collection"]}' if c else ""), pref,
            s.get("rerank_model") or "нет", (c["candidate_depth"] if c else s.get("candidate_depth")) or "-",
            f'{c["model"]}, {c["dimension"]}, RRF k = {c["rrf_k"]}, вес {num(c["weight"])}' if c else "нет"]
caption("Таблица 19 - Параметры пяти опытов второй версии")
md_table(["Опыт", "Коллекция Milvus", "Префиксы E5", "Реранкер", "Кандидатов для реранкера", "CLIP"],
         [params(e) for e in ["o1", "o2", "ctl", "o3", "o4", "o5"] if e in RUNS])

**Таблица 19 - Параметры пяти опытов второй версии**

| Опыт | Коллекция Milvus | Префиксы E5 | Реранкер | Кандидатов для реранкера | CLIP |
|---|---|---|---|---|---|
| 1. Без префиксов | diplom_v2_noprefix | нет | нет | - | нет |
| 2. Префиксы E5 | diplom_v2_base | query: / passage: | нет | - | нет |
| Контроль (повтор опыта 2) | diplom_v2_e5 | query: / passage: | нет | - | нет |
| 3. Реранкер, 20 | diplom_v2_rerank | query: / passage: | BAAI/bge-reranker-v2-m3 | 20 | нет |
| 4. Реранкер, 50 | diplom_v2_rerank | query: / passage: | BAAI/bge-reranker-v2-m3 | 50 | нет |
| 5. Реранкер, 50 + CLIP | diplom_v2_rerank + diplom_v2_clip | query: / passage: | BAAI/bge-reranker-v2-m3 | 50 | sentence-transformers/clip-ViT-B-32, 512, RRF k = 60, вес 1,0 |

Пояснения:

- Коллекция `diplom_v2_e5` имеет те же настройки, что `diplom_v2_base`. Её прогон оставлен как проверка воспроизводимости: независимая загрузка той же модели дала расхождение в 10 вопросах из 720 и в пределах 0,4 п.п. по всем метрикам на исходных вопросах. Это задаёт нижнюю границу различий, которые можно считать эффектом, а не случайным разбросом.
- Опыт 1 добавлен специально, чтобы префиксы E5 сравнивались с вариантом без них, а не с копией самого себя.
- Опыт 4 добавлен потому, что в опыте 5 с CLIP реранкер получал 50 кандидатов (глубина CLIP), а в опыте 3 - 20. Без опыта 4 эффект CLIP нельзя отделить от эффекта числа кандидатов.
- Ограничение max_length = 512 токенов для реранкера введено, потому что без него длинные чанки с таблицами при почти заполненной видеопамяти существенно замедляли переранжирование.
- Опыт 5 работает в три фазы: текстовый поиск на глубину 50, объединение с выдачей CLIP методом RRF (k = 60) и итоговое переранжирование 50 кандидатов.

Конфигурации: `V2/configs/pipeline_manual_merged_split.json` (опыты 2, 3, 5 и контроль) и `V2/configs/pipeline_manual_merged_split_extra.json` (опыты 1 и 4; результаты в `V2/results/retrieval/split_extra`).

In [6]:
# Проверка расхождения контрольного повтора с опытом 2 (раздел 11, пояснения)
diff = [q for q in D["o2"] if any(bool(D["o2"][q]["metrics"][k]["content"]["hit"]) != bool(D["ctl"][q]["metrics"][k]["content"]["hit"])
                                  for k in ("1", "3", "5", "10", "20"))]
mx = max(abs(agg("o2", "orig", k) - agg("ctl", "orig", k)) * 100 for k in ("1", "3", "5", "10", "20"))
print(f"Контроль и опыт 2: вопросов с разным попаданием (Hit@1-20) - {len(diff)} из 720; "
      f"наибольшая разница Hit@k на исходных вопросах - {num(mx)} п.п.")

Контроль и опыт 2: вопросов с разным попаданием (Hit@1-20) - 7 из 720; наибольшая разница Hit@k на исходных вопросах - 0,4 п.п.


## 12. Архитектура рабочего контура (прототип RAG)

### 12.1. Общая схема

Рабочий контур второй версии состоит из тех же моделей, что в первой (E5, реранкер `BAAI/bge-reranker-v2-m3` и языковая модель `qwen2.5:14b`), векторной базы Milvus и добавленной модели изображений CLIP. Подготовка корпуса выполняется командой `prepare-documents` и проверяется скриптами `check_extraction.py` (проверка извлечения) и `check_preparation.py` (проверка нарезки); ручные главы 1.1-1.9 подключаются импортёром `import_extracted_to_V2` и образуют корпус `V2/data/chunked_merged_manual`. Индексация (`load`, `load-clip`), поиск (`search`) и оценка (`evaluate`) запускаются одним скриптом `run_pipeline.py`; генерация ответов и их оценка описаны в разделе 14. Последовательность шагов показана на рисунке 12.

```
Тексты ПУЭ (data/raw, только чтение)
   ▼
prepare-documents + check_extraction.py / check_preparation.py; главы 1.1-1.9 - import_extracted_to_V2
   ▼
Корпус V2/data/chunked_merged_manual (один пункт - один чанк)
   ▼
load: multilingual-e5-base ("passage: "), Milvus AUTOINDEX, COSINE   |   load-clip: clip-ViT-B-32 (опыт 5)
   ▼
search: вопрос ("query: ", кроме опыта 1) - 20 или 50 кандидатов [+ CLIP, RRF k = 60]
   ▼
bge-reranker-v2-m3 (опыты 3-5): первые 5 фрагментов
   ▼
evaluate: Hit@k, MRR, nDCG@5, все факты в первых 5 (metrics.json)
   ▼
generate.py: qwen2.5:14b по 5 фрагментам   -   judge.py: оценка LLM-судьёй
```

**Рисунок 12 - Общая схема второй версии**

### 12.2. Реализованные модули (код)

**Таблица 20 - Реализованные модули (код)**

| Файл | Назначение |
|---|---|
| `V2/scripts/run_pipeline.py` | Командная строка: validate, prepare-documents, prepare, load, load-clip, search, evaluate, all |
| `V2/src/retrieval_eval/backend.py` | Загрузка эмбеддингов в Milvus, поиск, переранжирование, подпись индекса (модель, размерность, префиксы, нормировка, хеши чанков и ресурсов) |
| `V2/src/retrieval_eval/clip_backend.py` | Коллекция изображений CLIP, поиск по изображениям, слияние RRF |
| `V2/src/retrieval_eval/scoring.py` | Метрики Hit@k, MRR, nDCG, покрытие обязательных фактов |
| `V2/src/retrieval_eval/common.py` | Конфигурация, проверки, построчный вывод хода выполнения |
| `V2/scripts/build_split_eval/build.py` | Сборка набора из 720 вопросов и qrels из Excel |
| `scripts/import_extracted_to_V2/` | Импорт и объединение ручных глав 1.1-1.9 |

Защита данных: `--allow-db-write` создаёт коллекцию и останавливается, если она уже есть (ничего не удаляется); `--reuse-collection` использует готовую коллекцию и сверяет её подпись с файлом конфигурации.

### 12.3. Как считается метрика качества ранжированного поиска (Hit@k)

Для каждого вопроса берутся первые k чанков выдачи.

- **Hit@k** - доля вопросов, у которых среди первых k есть хотя бы один чанк с эталонным текстом (оценка 2).
- **MRR** - среднее 1/r, где r - позиция первого такого чанка в выдаче глубиной 20 (0, если не найден).
- **nDCG@5** - дисконтированная полезность первых пяти позиций относительно идеального порядка эталонных чанков.
- **Все факты в первых 5** - доля вопросов, у которых все обязательные абзацы эталона содержатся в первых пяти чанках. Это более строгая метрика: составной вопрос засчитывается, только если найдены все его части.

Изображения в зачёт пока не входят: оценка ведётся в режиме «только текст», поэтому опыт 5 оценивает влияние CLIP только на порядок текстовых чанков, а не на то, выдаёт ли система нужный рисунок.

## 13. Графическое подтверждение (метрики прототипа)

### 13.1. Итоговая таблица: 268 исходных вопросов

Результаты пяти опытов на 268 исходных вопросах приведены в таблице 21 и на рисунке 13.

In [7]:
def main_table(g):
    md_table(["Опыт", "Hit@1", "Hit@3", "Hit@5", "Hit@10", "Hit@20", "MRR", "nDCG@5", "Все факты в первых 5"],
             [[NAME[e]] + [pct(agg(e, g, k)) for k in ("1", "3", "5", "10", "20")]
              + [num(agg(e, g, "20", key="rr"), 3), num(agg(e, g, "5", key="ndcg_reference_set"), 3),
                 pct(agg(e, g, "5", "facts", "full_text_answer"))] for e in EXP])
caption("Таблица 21 - Итоговая таблица: 268 исходных вопросов")
main_table("orig")
bar("Рисунок 13 - Hit@k на 268 исходных вопросах", ["Hit@1", "Hit@3", "Hit@5", "Hit@10"],
    [(NAME[e], [agg(e, "orig", k) * 100 for k in ("1", "3", "5", "10")], COLOR[e]) for e in EXP], yrange=[50, 105])

**Таблица 21 - Итоговая таблица: 268 исходных вопросов**

| Опыт | Hit@1 | Hit@3 | Hit@5 | Hit@10 | Hit@20 | MRR | nDCG@5 | Все факты в первых 5 |
|---|---|---|---|---|---|---|---|---|
| Опыт 1. Без префиксов | 68,7 | 84,7 | 88,8 | 94,0 | 95,9 | 0,778 | 0,789 | 86,2 |
| Опыт 2. Префиксы E5 | 71,6 | 86,2 | 91,0 | 94,4 | 96,3 | 0,800 | 0,812 | 88,1 |
| Опыт 3. Реранкер, 20 | 88,1 | 95,1 | 95,5 | 96,3 | 96,3 | 0,915 | 0,913 | 92,5 |
| Опыт 4. Реранкер, 50 | 89,9 | 97,0 | 98,5 | 98,9 | 99,3 | 0,937 | 0,940 | 96,3 |
| Опыт 5. Реранкер, 50 + CLIP | 90,3 | 97,0 | 98,1 | 98,5 | 98,9 | 0,938 | 0,940 | 95,9 |

### 13.2. Итоговая таблица: все 720 вопросов

Результаты пяти опытов на всех 720 вопросах приведены в таблице 22.

In [8]:
caption("Таблица 22 - Итоговая таблица: все 720 вопросов")
main_table("all")

**Таблица 22 - Итоговая таблица: все 720 вопросов**

| Опыт | Hit@1 | Hit@3 | Hit@5 | Hit@10 | Hit@20 | MRR | nDCG@5 | Все факты в первых 5 |
|---|---|---|---|---|---|---|---|---|
| Опыт 1. Без префиксов | 62,8 | 83,8 | 88,1 | 93,3 | 95,3 | 0,739 | 0,764 | 86,8 |
| Опыт 2. Префиксы E5 | 67,1 | 83,5 | 89,4 | 93,5 | 95,6 | 0,767 | 0,788 | 88,1 |
| Опыт 3. Реранкер, 20 | 89,3 | 94,4 | 95,0 | 95,6 | 95,6 | 0,919 | 0,921 | 93,6 |
| Опыт 4. Реранкер, 50 | 90,8 | 96,4 | 97,5 | 98,1 | 98,2 | 0,938 | 0,943 | 96,4 |
| Опыт 5. Реранкер, 50 + CLIP | 91,1 | 96,2 | 97,2 | 97,9 | 98,1 | 0,939 | 0,942 | 96,1 |

### 13.3. Ранговые метрики: MRR и nDCG@5

Ранговые метрики MRR и nDCG@5 по пяти опытам показаны на рисунке 14.

In [9]:
bar("Рисунок 14 - MRR и nDCG@5", [SHORT[e] for e in EXP],
    [("MRR (глубина 20)", [agg(e, "orig", "20", key="rr") for e in EXP], "#5b8def"),
     ("nDCG@5", [agg(e, "orig", "5", key="ndcg_reference_set") for e in EXP], "#2a9d8f")], ytitle="", yrange=[0.7, 1.0], d=3)

### 13.4. Исходные вопросы и их части

Части X.1, X.2 … короче и уже исходных вопросов. Без реранкера они находятся реже исходных, с реранкером - не хуже. Результаты приведены в таблице 23 и на рисунках 15 и 16.

In [10]:
caption("Таблица 23 - Метрики поиска на 452 частях вопросов")
main_table("parts")
print(f'Опыт 2: Hit@1 частей {pct(agg("o2", "parts", "1"))} % против {pct(agg("o2", "orig", "1"))} % у исходных; '
      f'опыт 4: {pct(agg("o4", "parts", "1"))} % против {pct(agg("o4", "orig", "1"))} %.')
bar("Рисунок 15 - Hit@1: исходные вопросы и части", [SHORT[e] for e in EXP],
    [("исходные (268)", [agg(e, "orig", "1") * 100 for e in EXP], "#5b8def"),
     ("части X.1, X.2 … (452)", [agg(e, "parts", "1") * 100 for e in EXP], "#f2a541")], yrange=[55, 100])
bar("Рисунок 16 - Все обязательные факты в первых пяти фрагментах", [SHORT[e] for e in EXP],
    [("исходные (268)", [agg(e, "orig", "5", "facts", "full_text_answer") * 100 for e in EXP], "#5b8def"),
     ("все (720)", [agg(e, "all", "5", "facts", "full_text_answer") * 100 for e in EXP], "#2a9d8f")], yrange=[75, 100])

**Таблица 23 - Метрики поиска на 452 частях вопросов**

| Опыт | Hit@1 | Hit@3 | Hit@5 | Hit@10 | Hit@20 | MRR | nDCG@5 | Все факты в первых 5 |
|---|---|---|---|---|---|---|---|---|
| Опыт 1. Без префиксов | 59,3 | 83,2 | 87,6 | 92,9 | 94,9 | 0,717 | 0,749 | 87,2 |
| Опыт 2. Префиксы E5 | 64,4 | 81,9 | 88,5 | 92,9 | 95,1 | 0,747 | 0,774 | 88,1 |
| Опыт 3. Реранкер, 20 | 90,0 | 94,0 | 94,7 | 95,1 | 95,1 | 0,921 | 0,926 | 94,2 |
| Опыт 4. Реранкер, 50 | 91,4 | 96,0 | 96,9 | 97,6 | 97,6 | 0,938 | 0,944 | 96,5 |
| Опыт 5. Реранкер, 50 + CLIP | 91,6 | 95,8 | 96,7 | 97,6 | 97,6 | 0,939 | 0,944 | 96,2 |

Опыт 2: Hit@1 частей 64,4 % против 71,6 % у исходных; опыт 4: 91,4 % против 89,9 %.


### 13.5. Разбивка по разделам ПУЭ (Hit@5, число найденных из 268 исходных)

In [11]:
secs = sorted(sec)
def sec_hit(e, s, k="5"):
    ids = [q for q in D[e] if "." not in q and QS[q]["expected_clause"].split(".")[0] == s]
    return sum(D[e][q]["metrics"][k]["content"]["hit"] for q in ids), len(ids)
caption("Таблица 24 - Разбивка по разделам ПУЭ (Hit@5, число найденных из 268 исходных)")
md_table(["Раздел", "Вопросов"] + [SHORT[e] for e in EXP], [[s, sec[s]] + [sec_hit(e, s)[0] for e in EXP] for s in secs])
bar("Рисунок 17 - Hit@5 по разделам ПУЭ", [f"Раздел {s}" for s in secs],
    [(NAME[e], [sec_hit(e, s)[0] / sec_hit(e, s)[1] * 100 for s in secs], COLOR[e]) for e in EXP], yrange=[60, 105], d=0)

**Таблица 24 - Разбивка по разделам ПУЭ (Hit@5, число найденных из 268 исходных)**

| Раздел | Вопросов | Опыт 1 | Опыт 2 | Опыт 3 | Опыт 4 | Опыт 5 |
|---|---|---|---|---|---|---|
| 1 | 42 | 35 | 38 | 39 | 40 | 39 |
| 2 | 45 | 40 | 40 | 42 | 45 | 45 |
| 3 | 35 | 31 | 34 | 35 | 35 | 35 |
| 4 | 41 | 37 | 36 | 38 | 41 | 41 |
| 5 | 39 | 36 | 37 | 38 | 38 | 38 |
| 6 | 31 | 29 | 30 | 30 | 30 | 30 |
| 7 | 35 | 30 | 29 | 34 | 35 | 35 |

### 13.6. Сравнение с первым проектом (V1)

Данные первого проекта взяты из его файлов ранговых метрик `scripts/stage4_eval/results/retrieval/stage4_rank_results_base.json`, `..._e5.json` и `..._rerank.json` (268 вопросов, выдача глубиной 10). В первом проекте попаданием считался чанк эталонного пункта. Чтобы сравнение было корректным, для V2 здесь взят тот же уровень зачёта - по пункту (clause), а не по дословному абзацу. Вопросы те же 268 с теми же номерами (таблица 25, рисунок 18).

In [12]:
V1 = {}
for n, suf in ((1, "base"), (2, "e5"), (3, "rerank")):
    f = V1RES / f"stage4_rank_results_{suf}.json"
    if f.is_file(): V1[n] = json.loads(f.read_text(encoding="utf-8-sig"))
PAIR = [(1, "o1", "Без префиксов"), (2, "o2", "Префиксы E5"), (3, "o3", "Реранкер, 20 кандидатов")]
if len(V1) < 3:
    print("Файлы первого проекта не найдены в", V1RES, "- раздел пропущен")
else:
    cl = lambda e, k, key="hit": agg(e, "orig", k, "clause", key)
    caption("Таблица 25 - Сравнение первого (V1) и второго (V2) проектов, зачёт по пункту")
    md_table(["Опыт", "Hit@1 V1", "Hit@1 V2", "Hit@5 V1", "Hit@5 V2", "Hit@10 V1", "Hit@10 V2", "MRR@10 V1", "MRR@10 V2"],
             [[lab, pct(V1[n]["metrics"]["hit@1"]), pct(cl(e, "1")), pct(V1[n]["metrics"]["hit@5"]), pct(cl(e, "5")),
               pct(V1[n]["metrics"]["hit@10"]), pct(cl(e, "10")), num(V1[n]["metrics"]["mrr"], 3), num(cl(e, "10", "rr"), 3)]
              for n, e, lab in PAIR]
             + [["Лучший опыт V2 (реранкер, 50)", "-", pct(cl("o4", "1")), "-", pct(cl("o4", "5")), "-", pct(cl("o4", "10")),
                 "-", num(cl("o4", "10", "rr"), 3)]])
    fig = make_subplots(rows=1, cols=3, subplot_titles=["Hit@1", "Hit@5", "Hit@10"])
    for j, (k, kk) in enumerate((("1", "hit@1"), ("5", "hit@5"), ("10", "hit@10")), 1):
        for name, vals, color in (("V1, 2912 чанков", [V1[n]["metrics"][kk] * 100 for n, _, _ in PAIR], "#9aa5b1"),
                                  ("V2, 3146 чанков", [cl(e, k) * 100 for _, e, _ in PAIR], "#e4572e")):
            fig.add_bar(name=name, x=[lab for _, _, lab in PAIR], y=vals, marker_color=color, showlegend=(j == 1),
                        text=[num(v) for v in vals], textposition="outside", cliponaxis=False, legendgroup=name)
    fig.update_layout(barmode="group")
    _show(fig, "Рисунок 18 - Сравнение первого (V1) и второго (V2) проектов (зачёт по пункту)", "%", [50, 105], height=480)
    rows = []
    for n, e, lab in PAIR:
        v1 = {str(x["id"]): x["rank"] is not None and x["rank"] <= 5 for x in V1[n]["details"]}
        v2 = {q: bool(D[e][q]["metrics"]["5"]["clause"]["hit"]) for q in v1}
        rows.append([lab, sum(1 for q in v1 if v2[q] and not v1[q]), sum(1 for q in v1 if v1[q] and not v2[q])])
    print("По отдельным вопросам (Hit@5 по пункту):")
    md_table(["Опыт", "V2 нашёл, V1 нет", "V1 нашёл, V2 нет"], rows)

**Таблица 25 - Сравнение первого (V1) и второго (V2) проектов, зачёт по пункту**

| Опыт | Hit@1 V1 | Hit@1 V2 | Hit@5 V1 | Hit@5 V2 | Hit@10 V1 | Hit@10 V2 | MRR@10 V1 | MRR@10 V2 |
|---|---|---|---|---|---|---|---|---|
| Без префиксов | 58,2 | 68,7 | 81,3 | 88,8 | 87,7 | 94,0 | 0,681 | 0,776 |
| Префиксы E5 | 60,1 | 71,6 | 83,6 | 91,0 | 88,8 | 94,4 | 0,696 | 0,799 |
| Реранкер, 20 кандидатов | 72,0 | 87,7 | 88,8 | 95,5 | 92,9 | 96,3 | 0,793 | 0,913 |
| Лучший опыт V2 (реранкер, 50) | - | 89,6 | - | 98,5 | - | 98,9 | - | 0,935 |

По отдельным вопросам (Hit@5 по пункту):


| Опыт | V2 нашёл, V1 нет | V1 нашёл, V2 нет |
|---|---|---|
| Без префиксов | 35 | 15 |
| Префиксы E5 | 29 | 9 |
| Реранкер, 20 кандидатов | 26 | 8 |

Модели, их параметры и вопросы в парах одинаковы, поэтому прирост связан в основном с корпусом: ручные главы 1.1-1.9 и новая нарезка (3146 чанков вместо 2912). Полностью исключить другие отличия нельзя: в V2 другой индекс Milvus (AUTOINDEX вместо HNSW с ef = 64), а уровень «пункт» в V2 засчитывает чанки всех обязательных пунктов вопроса, а не только одного expected_clause. Для 268 вопросов строгий зачёт V2 по дословному абзацу (раздел 13.1) даёт почти те же числа, что зачёт по пункту.

Для сравнения с первыми прогонами серия прогонов первой версии (таблица 7, рисунок 2) продолжена на рисунке 19 основными опытами V1 и опытами V2 на тех же 268 вопросах. Значения V2 взяты при зачёте по пункту (таблица 25; Hit@3 - из тех же файлов `metrics.json`), то есть по той же схеме, что в V1. Прогоны 1.1-1.5 выполнены на наборах другого размера и состава (от 25 до 300 вопросов), поэтому напрямую сравнивать их с остальными нельзя: они показывают, как менялся результат по мере уточнения набора вопросов.

In [13]:
hr = V1RES / "stage4_hitk_runs.jsonl"
if hr.is_file() and len(V1) == 3:
    runs = [json.loads(l) for l in hr.read_text(encoding="utf-8-sig").splitlines() if l.strip()]
    series = []
    for r in runs:
        if series and series[-1]["n"] == r["n_questions"]: continue
        series.append({"n": r["n_questions"], "m": r["metrics_percent"]})
    cats = [f"1.{i} V1 (N={s['n']})" for i, s in enumerate(series[:-1], 1)]
    vals = {k: [s["m"][f"hit@{k}"] for s in series[:-1]] for k in ("1", "3", "5")}
    for n, lab in ((1, "Опыт 1 V1"), (2, "Опыт 2 V1"), (3, "Опыт 3 V1")):
        cats.append(lab)
        for k in vals: vals[k].append(V1[n]["metrics"][f"hit@{k}"] * 100)
    for e in EXP:
        cats.append(SHORT[e] + " V2")
        for k in vals: vals[k].append(agg(e, "orig", k, "clause") * 100)
    fig = go.Figure()
    for k, color in (("1", "#1f77b4"), ("3", "#ff7f0e"), ("5", "#2ca02c")):
        fig.add_scatter(name=f"Hit@{k}", x=cats, y=vals[k], mode="lines+markers", line=dict(color=color, width=2), marker=dict(size=8),
                        text=[num(v) for v in vals[k]], hovertemplate="%{x}<br>Hit@" + k + ": %{text}<extra></extra>")
    _show(fig, "Рисунок 19 - Серия прогонов V1 и опыты V2: Hit@1, Hit@3 и Hit@5 (зачёт по пункту)", "%", [0, 102], height=500)
    print(f'На 268 вопросах: Hit@5 {num(V1[1]["metrics"]["hit@5"] * 100)} % (опыт 1 V1) и {pct(agg("o4", "orig", "5", "clause"))} % (опыт 4 V2); '
          f'Hit@3 {num(V1[1]["metrics"]["hit@3"] * 100)} и {pct(agg("o4", "orig", "3", "clause"))} %; '
          f'Hit@1 {num(V1[1]["metrics"]["hit@1"] * 100)} и {pct(agg("o4", "orig", "1", "clause"))} %.')
else:
    print("Журнал прогонов V1 не найден:", hr)

На 268 вопросах: Hit@5 81,3 % (опыт 1 V1) и 98,5 % (опыт 4 V2); Hit@3 76,1 и 97,0 %; Hit@1 58,2 и 89,6 %.


### 13.7. Попарные изменения между опытами

Число исходных вопросов (из 268), которые перешли из «не найден» в «найден» и обратно при переходе от одного опыта к следующему (таблица 26).

In [14]:
def flips(a, b, k):
    ids = [q for q in D[a] if "." not in q]
    up = sum(1 for q in ids if not D[a][q]["metrics"][k]["content"]["hit"] and D[b][q]["metrics"][k]["content"]["hit"])
    dn = sum(1 for q in ids if D[a][q]["metrics"][k]["content"]["hit"] and not D[b][q]["metrics"][k]["content"]["hit"])
    return f"{up} / {dn}"
steps = [("o1", "o2", "От опыта 1 к опыту 2 (префиксы)"), ("o2", "o3", "От опыта 2 к опыту 3 (реранкер, 20)"),
         ("o3", "o4", "От опыта 3 к опыту 4 (50 кандидатов)"), ("o4", "o5", "От опыта 4 к опыту 5 (CLIP)"),
         ("o2", "ctl", "От опыта 2 к контрольному повтору")]
caption("Таблица 26 - Попарные изменения между опытами (исходные 268 вопросов)")
md_table(["Переход", "Hit@1: улучшено / ухудшено", "Hit@5: улучшено / ухудшено"],
         [[lab, flips(a, b, "1"), flips(a, b, "5")] for a, b, lab in steps if b in D])

**Таблица 26 - Попарные изменения между опытами (исходные 268 вопросов)**

| Переход | Hit@1: улучшено / ухудшено | Hit@5: улучшено / ухудшено |
|---|---|---|
| От опыта 1 к опыту 2 (префиксы) | 16 / 8 | 10 / 4 |
| От опыта 2 к опыту 3 (реранкер, 20) | 52 / 8 | 13 / 1 |
| От опыта 3 к опыту 4 (50 кандидатов) | 7 / 2 | 8 / 0 |
| От опыта 4 к опыту 5 (CLIP) | 1 / 0 | 0 / 1 |
| От опыта 2 к контрольному повтору | 0 / 1 | 1 / 0 |

### 13.8. Типовые отказы поиска

Вопросы, которые после опыта 4 или опыта 5 остаются вне первых пяти (таблица 27).

In [15]:
miss = [q for q in D["o5"] if "." not in q and (not D["o5"][q]["metrics"]["5"]["content"]["hit"] or not D["o4"][q]["metrics"]["5"]["content"]["hit"])]
def ref(q):
    extra = QS[q]["required_tables"] + QS[q]["required_figures"]
    return QS[q]["expected_clause"] + (f" ({', '.join(extra)})" if extra else "")
caption("Таблица 27 - Вопросы вне первых пяти после опытов 4 и 5")
md_table(["№", "Пункт эталона", "Вопрос", "Опыт 4", "Опыт 5"],
         [[q, ref(q), QS[q]["query"]] + ["найден" if D[e][q]["metrics"]["5"]["content"]["hit"] else "не найден" for e in ("o4", "o5")]
          for q in sorted(miss, key=int)])
never = [q for q in sorted(miss, key=int) if not any(D[e][q]["metrics"]["5"]["content"]["hit"] for e in EXP)]
print("Не попали в первые пять ни в одном из пяти опытов:", ", ".join(never))

**Таблица 27 - Вопросы вне первых пяти после опытов 4 и 5**

| № | Пункт эталона | Вопрос | Опыт 4 | Опыт 5 |
|---|---|---|---|---|
| 20 | 1.7.96 | Какие требования предъявляются к сопротивлению заземления в электроустановках напряжением выше 1 кВ с изолированной нейтралью? | найден | не найден |
| 24 | 1.7.82 (1.7.7) | Какие сторонние проводящие части подлежат присоединению к основной системе уравнивания потенциалов? | не найден | не найден |
| 35 | 1.8.17 (1.8.16, 1.8.33) | Какие испытания проводятся для измерительных трансформаторов тока? | не найден | не найден |
| 164 | 5.3.60 | Каковы требования к защите электродвигателей напряжением до 1 кВ от многофазных КЗ? | не найден | не найден |
| 206 | 6.1.11 | Каким образом производится выбор источников света для внутреннего и наружного освещения? | не найден | не найден |

Не попали в первые пять ни в одном из пяти опытов: 24, 164, 206


Вероятные причины (предположение по формулировкам, выдача по вопросам 24 и 35 приведена ниже): эталонный пункт опирается на рисунок или таблицу (вопрос 24, а также вопрос 35, не найденный в опытах 4 и 5) либо вопрос сформулирован в общем виде, и соседние пункты оказываются ближе к нему по словам (164, 206). Выдачи по этим вопросам сохранены в файлах выдачи поиска `retrieval.json` (первые 20 фрагментов по каждому вопросу) соответствующих прогонов; в пакет `data_V2` эти файлы не входят из-за размера (62-68 МБ каждый).

**Пример выдачи по вопросам вне первых пяти (вопросы 24 и 35).** Фрагменты, поданные модели в опыте 4, берутся из `answers.jsonl` (поле `contexts`), место эталонного пункта в первых 20 - из `metrics.json` каждого опыта (зачёт по пункту, `rr` на глубине 20).

In [16]:
_gd = sorted((GEN / "answers").glob("*_rerank50")) if (GEN / "answers").is_dir() else []
A138 = {}
if _gd and (_gd[-1] / "answers.jsonl").is_file():
    for _l in (_gd[-1] / "answers.jsonl").read_text(encoding="utf-8").splitlines():
        if _l.strip():
            _r = json.loads(_l); A138[str(_r["qid"])] = _r
def place(e, q):
    if e not in D or q not in D[e]: return "-"
    rr = D[e][q]["metrics"]["20"]["clause"]["rr"]
    return str(round(1 / rr)) if rr else "нет"
EX138 = ["24", "35"]
md_table(["№", "Вопрос", "Эталон", "Пункты первых пяти фрагментов (опыт 4)", "Место эталона в первых 20 по опытам"],
         [[q, QS[q]["query"], ref(q), ", ".join(c["clause"] for c in A138.get(q, {}).get("contexts", [])) or "-",
           "; ".join(f"{SHORT[e]}: {place(e, q)}" for e in ["o1", "o2", "ctl", "o3", "o4", "o5"])] for q in EX138])
for q in EX138:
    if q in A138:
        print(f"\nВопрос {q}. Ответ модели (опыт 4):\n{A138[q]['answer'].strip()}")

| № | Вопрос | Эталон | Пункты первых пяти фрагментов (опыт 4) | Место эталона в первых 20 по опытам |
|---|---|---|---|---|
| 24 | Какие сторонние проводящие части подлежат присоединению к основной системе уравнивания потенциалов? | 1.7.82 (1.7.7) | 1.7.83, 7.1.88, 7.1.87, 1.7.144, 1.7.133 | Опыт 1: нет; Опыт 2: нет; Контроль: нет; Опыт 3: нет; Опыт 4: нет; Опыт 5: нет |
| 35 | Какие испытания проводятся для измерительных трансформаторов тока? | 1.8.17 (1.8.16, 1.8.33) | 1.8.18, 1.8.25, 1.8.26, 1.8.29, 3.4.16 | Опыт 1: 5; Опыт 2: 3; Контроль: 3; Опыт 3: 6; Опыт 4: 7; Опыт 5: 7 |


Вопрос 24. Ответ модели (опыт 4):
Основная система уравнивания потенциалов должна соединять между собой следующие проводящие части согласно пункту 1.7.83 и пункту 7.1.87:

- основной (магистральный) защитный проводник;
- основной (магистральный) заземляющий проводник или основной заземляющий зажим;
- стальные трубы коммуникаций зданий и между зданиями;
- металлические части строительных конструкций, молниезащиты, системы центрального отопления, вентиляции и кондиционирования.

Для дополнительной системы уравнивания потенциалов, согласно пунктам 1.7.83 и 7.1.88, к ней должны быть подключены все одновременно доступные прикосновению открытые проводящие части стационарного электрооборудования, сторонние проводящие части, включая металлические части строительных конструкций здания, а также нулевые защитные проводники в системе TN и защитные заземляющие проводники в системах IT и TT, включая защитные проводники штепсельных розеток. В дополнение, в ванных и душевых помещениях должны быть под

Вопрос 24: пункт 1.7.82 (основная система уравнивания потенциалов, ссылка на рис. 1.7.7) не вошёл в первые 20 фрагментов ни в одном опыте; выше оказались пункты о дополнительной системе уравнивания потенциалов (1.7.83, 7.1.88) и о системе уравнивания на вводе в здание (7.1.87), и модель составила ответ по 1.7.83 и 7.1.87. Вопрос 35: фрагмент пункта 1.8.17 начинается с заголовка «Измерительные трансформаторы тока»; он стоит на 3-м месте в опыте 2 и контрольном повторе, на 5-м в опыте 1 и на 6-7-м после переранжирования (опыты 3-5), а первым идёт соседний пункт 1.8.18 о трансформаторах напряжения. Модель ответила, что для трансформаторов тока пунктов в контексте нет.

## 14. Генерация ответов по выдаче опыта 4 и их оценка

Генерация выполнена скриптом `V2/scripts/generation/generate.py`. На вход подавалась сохранённая выдача опыта 4 (E5 с префиксами, реранкер `bge-reranker-v2-m3` по 50 кандидатам), поэтому Milvus при генерации не нужен. Ответы получены на все 720 вопросов: 268 исходных и 452 части. Результаты: `V2/results/generation/answers/<папка>_rerank50/run.json` (все параметры и контрольные суммы входных файлов) и `answers.jsonl` (ответ, поданные фрагменты и служебные отметки по каждому вопросу).

### 14.1. Параметры генерации

In [17]:
def last_dir(kind, suffix):
    base = GEN / kind
    ds = sorted(base.glob("*_" + suffix), key=lambda p: p.name) if base.is_dir() else []
    return ds[-1] if ds else None
def read_jsonl(p):
    last = {}
    for l in p.read_text(encoding="utf-8").splitlines():
        if l.strip():
            r = json.loads(l); last[str(r["qid"])] = r   # при --resume берётся последняя строка по вопросу
    return list(last.values())
GDIR = last_dir("answers", "rerank50")
if GDIR is None:
    print("Прогон генерации не найден: нет папки V2/results/generation/answers/*_rerank50"); ANS = []; ALL = []
else:
    GRUN = json.loads((GDIR / "run.json").read_text(encoding="utf-8"))["settings"]
    ALL = read_jsonl(GDIR / "answers.jsonl")
    ANS = [r for r in ALL if not r.get("error")]
    print("Папка:", GDIR.relative_to(ROOT).as_posix(), "| вход:", GRUN["input"])
    caption("Таблица 28 - Параметры генерации")
    md_table(["Параметр", "Значение"], [
        ["Модель", f'{GRUN["model"]} (Ollama, {GRUN["host"]})'],
        ["Системная подсказка", "как в первом проекте (src/rag/rag_service.py, PUE_SYSTEM), без изменений"],
        ["Что подаётся модели", f'только текст вопроса и первые {GRUN["top_k"]} фрагментов выдачи; эталонный ответ и обязательные факты не подаются'],
        ["temperature", f'{num(GRUN["temperature"])} (как в первом проекте)'],
        ["num_predict (лимит ответа)", f'{GRUN["num_predict"]} токенов (как в первом проекте)'],
        ["num_ctx (окно модели)", f'{GRUN["num_ctx"]} токена; в первом проекте не задавалось (использовалось значение Ollama по умолчанию)'],
        ["Обрезка фрагмента", f'фрагмент длиннее {GRUN["max_chars"]} символов обрезается с пометкой'],
        ["Кандидатов для реранкера", GRUN["search_settings"]["mode"].get("candidate_depth")]])

Папка: V2/results/generation/answers/20261113_001359_rerank50 | вход: V2\results\retrieval\split_extra\20261111_000608_110945\retrieval.json


**Таблица 28 - Параметры генерации**

| Параметр | Значение |
|---|---|
| Модель | qwen2.5:14b-instruct-q4_K_M (Ollama, http://localhost:11434) |
| Системная подсказка | как в первом проекте (src/rag/rag_service.py, PUE_SYSTEM), без изменений |
| Что подаётся модели | только текст вопроса и первые 5 фрагментов выдачи; эталонный ответ и обязательные факты не подаются |
| temperature | 0,2 (как в первом проекте) |
| num_predict (лимит ответа) | 512 токенов (как в первом проекте) |
| num_ctx (окно модели) | 8192 токена; в первом проекте не задавалось (использовалось значение Ollama по умолчанию) |
| Обрезка фрагмента | фрагмент длиннее 4000 символов обрезается с пометкой |
| Кандидатов для реранкера | 50 |

### 14.2. Результаты генерации

Все числа посчитаны по файлу ответов модели `answers.jsonl` (одна строка на вопрос: ответ, поданные фрагменты и служебные отметки). Время - полное время ответа модели на вопрос (без поиска: выдача взята готовой). Результаты приведены в таблице 29 и на рисунках 20 и 21.

In [18]:
CL = re.compile(r"\b\d\.\d{1,2}\.\d{1,3}\b")
REF = re.compile(r"(в (приведённом|приведенном|предоставленном|данном) контексте (нет|не)|контекст(е)? не содержит|нет (информации|сведений|данных)|информаци\w* (нет|отсутству)|не указан\w* в (контексте|приведённых))", re.I)
def gstat(G):
    n = len(G) or 1
    return {"n": len(G), "el_mean": st.mean(r["elapsed_s"] for r in G), "el_med": st.median(r["elapsed_s"] for r in G),
            "pt": st.mean(r["prompt_tokens"] for r in G), "at": st.mean(r["answer_tokens"] for r in G),
            "clause": sum(bool(CL.search(r["answer"])) for r in G), "share": sum(bool(CL.search(r["answer"])) for r in G) / n,
            "notctx": sum(bool(set(CL.findall(r["answer"])) - {c["clause"] for c in r["contexts"]}) for r in G),
            "refused": [r["qid"] for r in G if REF.search(r["answer"])],
            "limit": [r["qid"] for r in G if r.get("answer_hit_limit")],
            "trunc": sum(any(c["truncated"] for c in r["contexts"]) for r in G)}
if ANS:
    GG = {"Все 720": ANS, "Исходные 268": [r for r in ANS if r["is_original"]], "Части 452": [r for r in ANS if not r["is_original"]]}
    GS = {k: gstat(v) for k, v in GG.items()}
    caption("Таблица 29 - Результаты генерации ответов")
    md_table(["Показатель", *GS], [
        ["Ответов получено / ошибок", *[f'{s["n"]} / {sum(1 for r in ALL if r.get("error"))}' if k == "Все 720" else f'{s["n"]} / 0' for k, s in GS.items()]],
        ["Время ответа, с: среднее", *[num(s["el_mean"]) for s in GS.values()]],
        ["Время ответа, с: медиана", *[num(s["el_med"]) for s in GS.values()]],
        ["Токенов в запросе (среднее)", *[round(s["pt"]) for s in GS.values()]],
        ["Токенов в ответе (среднее)", *[round(s["at"]) for s in GS.values()]],
        ["Ответ содержит номер пункта ПУЭ", *[f'{s["clause"]} ({pct(s["share"])} %)' for s in GS.values()]],
        ["Упомянут пункт, которого нет в заголовках поданных фрагментов", *[s["notctx"] for s in GS.values()]],
        ["Явный отказ («в контексте нет информации»)", *[len(s["refused"]) for s in GS.values()]],
        ["Ответ достиг предела 512 токенов", *[len(s["limit"]) for s in GS.values()]],
        ["Среди поданных фрагментов есть обрезанный (> 4000 символов)", *[s["trunc"] for s in GS.values()]]])
    print(f'Общее время генерации - {num(sum(r["elapsed_s"] for r in ANS) / 60)} мин, скорость модели - '
          f'{num(st.mean(r["tokens_per_s"] for r in ANS if r.get("tokens_per_s")))} токена/с в среднем; '
          f'максимум токенов в запросе - {max(r["prompt_tokens"] for r in ANS)} при окне {GRUN["num_ctx"]}.')
    fig = go.Figure()
    for k, color in [("Исходные 268", "#e4572e"), ("Части 452", "#5b8def")]:
        fig.add_histogram(x=[r["elapsed_s"] for r in GG[k]], name=k, marker_color=color, opacity=0.75, xbins=dict(size=1))
    fig.update_layout(barmode="overlay", xaxis_title="секунд")
    _show(fig, "Рисунок 20 - Время ответа модели: исходные вопросы и части", "вопросов")
    marks = [("Пункт не из поданных фрагментов", "notctx"), ("Явный отказ", "refused"), ("Предел 512 токенов", "limit"), ("Обрезанный фрагмент", "trunc")]
    val = lambda s, k: len(s[k]) if isinstance(s[k], list) else s[k]
    bar("Рисунок 21 - Отметки в ответах, требующие проверки", [m[0] for m in marks],
        [(k, [val(GS[k], m[1]) for m in marks], c) for k, c in (("Исходные 268", "#e4572e"), ("Части 452", "#5b8def"))], ytitle="ответов", d=0)

**Таблица 29 - Результаты генерации ответов**

| Показатель | Все 720 | Исходные 268 | Части 452 |
|---|---|---|---|
| Ответов получено / ошибок | 720 / 0 | 268 / 0 | 452 / 0 |
| Время ответа, с: среднее | 5,2 | 6,3 | 4,6 |
| Время ответа, с: медиана | 4,3 | 5,1 | 4,0 |
| Токенов в запросе (среднее) | 1917 | 1755 | 2014 |
| Токенов в ответе (среднее) | 143 | 177 | 123 |
| Ответ содержит номер пункта ПУЭ | 706 (98,1 %) | 267 (99,6 %) | 439 (97,1 %) |
| Упомянут пункт, которого нет в заголовках поданных фрагментов | 70 | 33 | 37 |
| Явный отказ («в контексте нет информации») | 5 | 2 | 3 |
| Ответ достиг предела 512 токенов | 10 | 6 | 4 |
| Среди поданных фрагментов есть обрезанный (> 4000 символов) | 69 | 24 | 45 |

Общее время генерации - 62,8 мин, скорость модели - 37,5 токена/с в среднем; максимум токенов в запросе - 6999 при окне 8192.


### 14.3. Наблюдения

**Отказы.** Модель ответила, что в поданных фрагментах нужной информации нет (список ниже). Вопросы 164 и 206 - те же, которые, согласно разделу 13.8, не попали в первые пять ни в одном опыте поиска, то есть отказ здесь вызван поиском, а не генерацией, и модель не сформировала ответ без опоры на фрагменты.

**Длинные ответы.** Ответы, достигшие 512 токенов, могли быть обрезаны. Лимит оставлен как в первом проекте ради сравнимости; при необходимости возможен отдельный прогон с num_predict = 1024.

**Ссылки на пункты.** Пункт, которого нет среди номеров пунктов пяти поданных фрагментов, может быть ссылкой из текста самого фрагмента (фрагменты часто ссылаются на другие пункты) или ошибкой модели; без проверки по тексту фрагментов это не считается ошибкой. Проверка выполняется на этапе оценки судьёй.

In [19]:
if ANS:
    print("Явный отказ:", ", ".join(GS["Все 720"]["refused"]) or "нет")
    print("Достигли предела 512 токенов:", ", ".join(GS["Все 720"]["limit"]) or "нет")

Явный отказ: 71.12, 72.8, 82.3, 164, 206
Достигли предела 512 токенов: 7, 21, 49, 78.4, 92, 115.1, 130, 130.2, 188.4, 190


### 14.4. Оценка ответов LLM-судьёй

Оценка выполнена скриптом `V2/scripts/generation/judge.py` на всех 720 ответах раздела 14.2. Судья - та же модель qwen2.5:14b-instruct-q4_K_M, temperature 0, ответ в формате JSON. В отличие от отвечающей модели, судья получает эталон: обязательные факты вопроса, а также сам вопрос, пять поданных фрагментов и ответ. Каждый из пяти критериев оценивается баллом 0, 1 или 2; в таблице - средний балл, делённый на 2 (1 = максимум). Результаты: `V2/results/generation/judge/<папка>_rerank50/judgments.jsonl` (оценка и комментарий по каждому ответу) и `judge_metrics.json` (сводка). Результаты приведены в таблице 30 и на рисунках 22 и 23.

In [20]:
CRIT = [("facts", "Факты (соответствие обязательным фактам)"), ("numbers", "Числа и единицы"), ("exceptions", "Исключения и условия"),
        ("sources", "Подтверждение источником (пункт)"), ("refusal", "Обоснованность отказа")]
CS = {"facts": "Факты", "numbers": "Числа", "exceptions": "Исключения", "sources": "Источник", "refusal": "Отказ"}
JDIR = last_dir("judge", "rerank50")
if JDIR is None or not (JDIR / "judge_metrics.json").is_file():
    print("Оценок судьи нет: не найдена папка V2/results/generation/judge/*_rerank50"); JM = None; JR = []
else:
    JM = json.loads((JDIR / "judge_metrics.json").read_text(encoding="utf-8"))
    JR = read_jsonl(JDIR / "judgments.jsonl") if (JDIR / "judgments.jsonl").is_file() else []
    print("Папка:", JDIR.relative_to(ROOT).as_posix(), "| оценено:", JM["n_judged"], "из", JM["n_answers"], "| сбоев:", len(JM["failed_qids"]))
    A = JM["answerable"]; G3 = ["all", "originals", "parts"]
    rate = lambda g, key: pct(round(A[g][key] * A[g]["n"]) / A[g]["n"]) + " %"   # доля по числу ответов (в JSON округлена до 4 знаков)
    cnt = lambda g, key: f'{round(A[g][key] * A[g]["n"])} ({pct(A[g][key])} %)'
    caption("Таблица 30 - Оценка ответов LLM-судьёй")
    md_table(["Критерий", "Все 720", "Исходные 268", "Части 452"],
             [[c[1], *[pct(A[g][c[0]]) + " %" for g in G3]] for c in CRIT] + [
              ["Итоговый балл (среднее пяти)", *[pct(A[g]["total"]) + " %" for g in G3]],
              ["Все пять критериев на максимуме (2)", *[rate(g, "all_criteria_2") for g in G3]],
              ["Все пять критериев не ниже 1", *[rate(g, "all_criteria_ge1") for g in G3]],
              ["Критическая ошибка (по судье)", *[cnt(g, "critical_error_rate") for g in G3]],
              ["Отказ (по судье)", *[cnt(g, "refused_rate_judge") for g in G3]],
              ["Эталонный пункт назван в ответе (автоматически)", *[rate(g, "clause_in_answer") for g in G3]],
              ["Эталонный пункт среди поданных фрагментов (автоматически)", *[rate(g, "clause_in_context") for g in G3]]])
    if A.get("development") and A.get("reserved_evaluation"):
        dv, rv = A["development"], A["reserved_evaluation"]
        print(f'По разбиению набора: разработка ({dv["n"]}) - {pct(dv["total"])} %, отложенные ({rv["n"]}) - {pct(rv["total"])} %; '
              f'разница {num(abs(dv["total"] - rv["total"]) * 100)} п.п.')
    bar("Рисунок 22 - Оценка судьи по пяти критериям: исходные вопросы и части", [CS[c[0]] for c in CRIT] + ["Итог"],
        [(n, [100 * A[g][c] for c in [x[0] for x in CRIT] + ["total"]], col) for g, n, col in
         (("originals", "Исходные 268", "#e4572e"), ("parts", "Части 452", "#5b8def"))], yrange=[70, 102])
if JR:
    tot = lambda r: sum(r["judge"][c] for c, _ in CRIT)
    fig = go.Figure()
    for k, flag, color in [("Исходные 268", True, "#e4572e"), ("Части 452", False, "#5b8def")]:
        c = Counter(tot(r) for r in JR if r["kind"] == "answerable" and r["is_original"] == flag)
        fig.add_bar(x=list(range(11)), y=[c.get(s, 0) for s in range(11)], name=k, marker_color=color, text=[c.get(s, 0) for s in range(11)])
    fig.update_layout(barmode="group", xaxis_title="сумма баллов (0-10)")
    _show(fig, "Рисунок 23 - Распределение суммы баллов судьи на ответ", "ответов")

Папка: V2/results/generation/judge/20261113_204638_rerank50 | оценено: 720 из 720 | сбоев: 0


**Таблица 30 - Оценка ответов LLM-судьёй**

| Критерий | Все 720 | Исходные 268 | Части 452 |
|---|---|---|---|
| Факты (соответствие обязательным фактам) | 92,8 % | 90,5 % | 94,2 % |
| Числа и единицы | 92,2 % | 91,4 % | 92,7 % |
| Исключения и условия | 90,8 % | 89,7 % | 91,4 % |
| Подтверждение источником (пункт) | 95,0 % | 94,6 % | 95,2 % |
| Обоснованность отказа | 95,8 % | 95,7 % | 95,9 % |
| Итоговый балл (среднее пяти) | 93,3 % | 92,4 % | 93,9 % |
| Все пять критериев на максимуме (2) | 85,1 % | 81,7 % | 87,2 % |
| Все пять критериев не ниже 1 | 94,6 % | 95,1 % | 94,2 % |
| Критическая ошибка (по судье) | 19 (2,6 %) | 7 (2,6 %) | 12 (2,6 %) |
| Отказ (по судье) | 9 (1,2 %) | 4 (1,5 %) | 5 (1,1 %) |
| Эталонный пункт назван в ответе (автоматически) | 93,2 % | 93,7 % | 92,9 % |
| Эталонный пункт среди поданных фрагментов (автоматически) | 97,5 % | 98,5 % | 96,9 % |

По разбиению набора: разработка (571) - 93,6 %, отложенные (149) - 92,2 %; разница 1,4 п.п.


Критические ошибки по судье (список ниже). Если эталонного пункта нет среди заголовков пяти поданных фрагментов, ошибка в основном вызвана поиском; если пункт был подан - это ошибка генерации (неверное число или категория, пропуск запрета, отказ при наличии пункта). Часть отметок судьи, вероятно, ошибочна: например, у 5.3 он поставил за факты и числа 2 из 2 и одновременно отметил критическую ошибку. Ручная проверка этих ответов пока не выполнялась.

In [21]:
if JR:
    crit_rows = sorted([r for r in JR if r["kind"] == "answerable" and r["judge"].get("critical_error")],
                       key=lambda r: [int(x) for x in r["qid"].split(".")])
    print("Критические ошибки:", len(crit_rows), "| эталонный пункт не подан поиском:",
          ", ".join(r["qid"] for r in crit_rows if not r["auto"]["required_clause_in_context"]),
          "| подан:", ", ".join(r["qid"] for r in crit_rows if r["auto"]["required_clause_in_context"]))
    md_table(["№", "Группа", "Пункт во фрагментах", "Баллы (ф/ч/и/и/о)", "Комментарий судьи"],
             [[r["qid"], "исходный" if r["is_original"] else "часть", "да" if r["auto"]["required_clause_in_context"] else "нет",
               "/".join(str(r["judge"][c]) for c, _ in CRIT), r["judge"].get("comment", "").replace("|", "/").replace("\n", " ")[:220]]
              for r in crit_rows])

Критические ошибки: 19 | эталонный пункт не подан поиском: 7.4, 24, 35, 71.11, 71.13, 78.7, 115.2, 164, 206 | подан: 5.3, 34.5, 56.7, 129.2, 129.6, 149, 175, 202.1, 202.2, 246


| № | Группа | Пункт во фрагментах | Баллы (ф/ч/и/и/о) | Комментарий судьи |
|---|---|---|---|---|
| 5.3 | часть | да | 2/2/2/1/2 | Ответ модели противоречит ПУЭ, так как в ПУЭ нет упоминания о понижении перегрузок для кабелей, находящихся в эксплуатации более 15 лет. В ответе указано неверное утверждение, что это указано в таблице 1.3.2, что не соот |
| 7.4 | часть | нет | 0/0/0/0/0 | Ответ модели противоречит ПУЭ, так как по режиму КЗ провода ВЛ напряжением выше 1 кВ не проверяются, согласно п. 1.4.3. |
| 24 | исходный | нет | 0/0/0/0/0 | Ответ модели не передает обязательные факты из пункта 1.7.82 ПУЭ и вместо этого ссылается на другие пункты, не относящиеся к вопросу. В ответе отсутствуют ключевые элементы, такие как металлические трубы коммуникаций, ме |
| 34.5 | часть | да | 1/0/0/0/0 | Ответ не передает всех обязательных фактов, приведенных в эталоне. В частности, пропущены важные пункты 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15 и 16. Кроме того, ответ содержит ошибку, так как утверждает, что испытание  |
| 35 | исходный | нет | 0/0/0/0/2 | Модель неверно указала, что специфические пункты для измерительных трансформаторов тока не приведены в предоставленном контексте, хотя они есть в пункте 1.8.17. Модель должна была указать конкретные испытания, приведенны |
| 56.7 | часть | да | 0/0/0/0/0 | Модель неверно указала пункт ПУЭ, где указано требование. Вместо 2.3.113 указан 2.3.133, что не соответствует обязательным фактам. |
| 71.11 | часть | нет | 0/0/0/0/0 | Ответ модели не передает определение штыревого изолятора из пункта 2.5.3 ПУЭ и вместо этого ссылается на контексты применения, не предоставляя определения. |
| 71.13 | часть | нет | 0/0/0/0/0 | Ответ модели не передает обязательные факты и содержит противоречивую информацию о пляске проводов, не соответствующую ПУЭ. |
| 78.7 | часть | нет | 0/0/0/0/2 | Модель неверно отказаласlica в ответе, не указав, что в ПУЭ есть пункт 2.5.207, который прямо указывает на необходимость вырубки деревьев, угрожающих падением на провода или опоры ВЛ. |
| 115.2 | часть | нет | 2/0/0/1/0 | Ответ модели передает смысл обязательных фактов, но не содержит числовых значений, которые присутствуют в эталоне. Кроме того, модель не указала номер пункта 3.4.4 ПУЭ, который является ключевым для ответа на вопрос. |
| 129.2 | часть | да | 1/0/0/1/0 | Ответ модели содержит ошибку в передаче числовых значений и объема маслоприемника без отвода масла. В ответе указано неверное значение для маслоприемника с отводом масла, а также отсутствует информация о расчете объема м |
| 129.6 | часть | да | 1/0/0/1/2 | Ответ модели не передает смысл обязательных фактов из пункта 4.2.69, который требует обязательного устройства маслоотвода при установке маслонаполненного оборудования на железобетонном перекрытии здания. Модель не указал |
| 149 | исходный | да | 0/0/0/0/0 | Модель привела определение из другого пункта, не соответствующее эталону. Это противоречит нормативному определению и содержит критическую ошибку. |
| 164 | исходный | нет | 0/0/0/0/0 | Ответ модели не содержит информации, относящейся к вопросу, и не упоминает пункт 5.3.60 ПУЭ, который является ключевым для ответа на вопрос. Вместо этого модель ссылается на неправильные разделы ПУЭ. |
| 175 | исходный | да | 0/0/0/0/1 | Модель неверно отказала, не указав, что приведённые требования в п. 5.5.10 полностью соответствуют вопросу и содержат все обязательные факты. |
| 202.1 | часть | да | 1/1/1/1/2 | Ответ противоречит пункту 6.1.27 ПУЭ, утверждая допустимость применения общих групповых щитков для освещения безопасности и эвакуационного освещения. |
| 202.2 | часть | да | 1/2/0/1/2 | Ответ модели противоречит пункту 6.1.27 ПУЭ, запрещающему применение общих групповых щитков для освещения безопасности и эвакуационного освещения. В ответе модель допускает питание от общих щитков, что неверно. |
| 206 | исходный | нет | 0/0/0/0/0 | Ответ модели не передает обязательные факты из ПУЭ, а вместо этого ссылается на отсутствие информации в предоставленных фрагментах, хотя эти факты приведены в эталоне. |
| 246 | исходный | да | 1/0/0/1/0 | Ответ модели содержит критическую ошибку: водород относится к категории IIС, но модель указала IIС и Т1, что неверно. Водород относится к категории IIС и группе Т6. |

**Пример автоматической проверки: номер пункта в ответе, которого нет среди поданных фрагментов** (`judgments.jsonl`, поле `auto.clauses_in_answer_not_in_context`). Номера ищутся регулярным выражением `CLAUSE_RE` (`V2/scripts/generation/gen_common.py`, функция `clauses_in`) и в `judge.py` вычитаются пункты поданных фрагментов и номера, упомянутые в их тексте; текст фрагментов для судьи обрезан до 1500 символов (`--max-chars 1500`), модели подавалось до 4000.

In [22]:
if JR:
    _ac = [r for r in JR if r["kind"] == "answerable" and r["auto"]["clauses_in_answer_not_in_context"]]
    print(f"Проверка сработала на {len(_ac)} ответах из {len(JR)}")
    _AB = {str(r["qid"]): r for r in ANS}
    md_table(["№", "Вопрос", "Эталон", "Пункты поданных фрагментов", "Номер в ответе вне фрагментов", "Судья (ф/ч/и/и/о)"],
             [[r["qid"], QS[r["qid"]]["query"], ", ".join(QS[r["qid"]]["required_clauses"]),
               ", ".join(c["clause"] for c in _AB.get(r["qid"], {}).get("contexts", [])),
               ", ".join(r["auto"]["clauses_in_answer_not_in_context"]), "/".join(str(r["judge"][c]) for c, _ in CRIT)]
              for r in sorted(_ac, key=lambda r: [int(x) for x in r["qid"].split(".")])])

Проверка сработала на 5 ответах из 720


| № | Вопрос | Эталон | Пункты поданных фрагментов | Номер в ответе вне фрагментов | Судья (ф/ч/и/и/о) |
|---|---|---|---|---|---|
| 37 | Какие испытательные напряжения промышленной частоты и длительность испытания установлены для первичных цепей смонтированных ячеек КРУ и КРУН классов 6 и 10 кВ с керамической изоляцией? | 1.8.25 | 1.8.25, 1.8.26, 1.8.19, 1.8.35, 1.8.20 | 1.8.22 | 2/2/2/2/2 |
| 38.5 | Каким значениям должно удовлетворять сопротивление заземляющих устройств при приемо-сдаточных испытаниях? | 1.8.39 | 1.7.104, 1.7.163, 1.8.39, 1.7.56, 1.7.101 | 1.8.38 | 1/1/1/1/2 |
| 130.3 | Распространяются ли требования п. 4.2.68 на КТП наружной установки? | 4.2.68 | 7.2.18, 7.3.84, 4.2.122, 4.2.203, 4.2.71 | 4.2.68 | 1/2/1/1/1 |
| 216 | Каковы особенности питания светильников наружного уличного освещения от сетей ВЛ до 1 кВ? | 6.3.16 | 6.3.16, 6.3.3, 6.3.18, 6.3.2, 6.1.17 | 6.3.4 | 2/2/2/2/2 |
| 236 | Какие наименьшие сечения медных жил установлены для групповых линий, линий от этажного до квартирного щитка и стояков жилого здания? | 7.1.34 | 7.1.34, 7.1.22, 7.1.36, 3.4.12, 2.1.14 | 7.1.1 | 2/2/2/2/2 |

Ручной разбор: в 37, 38.5 и 236 модель назвала таблицу (1.8.22, 1.8.38, 7.1.1), ссылка на которую стоит в поданном пункте (1.8.25, 1.8.39, 7.1.34) сразу за границей обрезки в 1500 символов (1505-1668-й символ); в 130.3 номер 4.2.68 взят из самого вопроса. Это ложные срабатывания. Ссылка на непереданный пункт есть только в ответе 216 (6.3.4), причём модель сама оговорила, что этого пункта в контексте нет; судья поставил за ответ 2 по всем критериям.

### 14.5. Проверка правильного отказа: вопросы без ответа в ПУЭ

В основном наборе у всех вопросов есть нормативный источник, поэтому отдельно проверено, не формирует ли модель ответ, когда его в ПУЭ нет. Составлен набор из 48 вопросов (`V2/data/rag_eval_questions/eval_questions_refusal.json`, R01-R48, answerability = not_in_pue) в четырёх категориях: вопросы, регулируемые другими документами; технологии, которых нет в ПУЭ 7-го издания; смежные темы, где тема в корпусе упомянута, но запрошенного значения нет; вопросы вне предметной области. Поиск выполнен той же конфигурацией, что опыт 4 (`V2/scripts/generation/refusal_search.py`), генерация - с теми же параметрами, что в разделе 14.1, оценка - судьёй. Судья ставит refusal_correct: 2 - ясный отказ без вымышленной нормы; 1 - отказ с посторонними пунктами, поданными как ответ; 0 - ответ со ссылкой на несуществующую норму. При ручной проверке засчитывается только отказ без посторонних пунктов; её результат задан списком `MANUAL_NOT_OK` ниже. Результаты приведены в таблице 31 и на рисунке 24.

In [23]:
MANUAL_NOT_OK = {"R17", "R22", "R26", "R28", "R32"}   # ручная проверка (раздел 14.5 отчёта)
CATN = {"other_norm": "другие нормы", "new_technology": "новые технологии", "partial_context": "смежные темы", "out_of_domain": "вне темы"}
RJDIR, RADIR = last_dir("judge", "refusal"), last_dir("answers", "refusal")
if RJDIR is None or not (RJDIR / "judgments.jsonl").is_file():
    print("Оценок набора на отказ нет: не найдена папка V2/results/generation/judge/*_refusal"); RJ = []
else:
    RM = json.loads((RJDIR / "judge_metrics.json").read_text(encoding="utf-8"))["not_in_pue"]
    RJ = [r for r in read_jsonl(RJDIR / "judgments.jsonl") if r["kind"] == "not_in_pue"]
    RA = {str(r["qid"]): r for r in read_jsonl(RADIR / "answers.jsonl")} if RADIR else {}
    print("Папка:", RJDIR.relative_to(ROOT).as_posix(), "| оценено:", RM["judged"], "из", RM["n"])
    def cell(a, n): return f"{a} ({num(100 * a / n)} %)"
    rows, chart = [], []
    for c, name in CATN.items():
        g = [r for r in RJ if r["category"] == c]
        if g:
            j = sum(r["judge"]["refusal_correct"] == 2 for r in g); m = sum(r["qid"] not in MANUAL_NOT_OK for r in g)
            rows.append([name, len(g), cell(j, len(g)), cell(m, len(g))]); chart.append((name, 100 * j / len(g), 100 * m / len(g)))
    j = sum(r["judge"]["refusal_correct"] == 2 for r in RJ); m = sum(r["qid"] not in MANUAL_NOT_OK for r in RJ)
    rows.append(["Всего", len(RJ), cell(j, len(RJ)), cell(m, len(RJ))])
    caption("Таблица 31 - Правильный отказ по категориям вопросов")
    md_table(["Категория", "Вопросов", "Правильный отказ (судья)", "Правильный отказ (ручная проверка)"], rows)
    bar("Рисунок 24 - Правильный отказ: судья и ручная проверка", [r[0] for r in chart],
        [("судья", [r[1] for r in chart], "#5b8def"), ("ручная проверка", [r[2] for r in chart], "#e4572e")], yrange=[0, 115], d=0)
    print(f'По судье: ответил по существу вместо отказа - {pct(RM["answered_instead_rate"])} %, вымышленный ответ - {pct(RM["hallucination_rate"])} %.')

Папка: V2/results/generation/judge/20261113_224440_refusal | оценено: 48 из 48


**Таблица 31 - Правильный отказ по категориям вопросов**

| Категория | Вопросов | Правильный отказ (судья) | Правильный отказ (ручная проверка) |
|---|---|---|---|
| другие нормы | 16 | 16 (100,0 %) | 16 (100,0 %) |
| новые технологии | 14 | 14 (100,0 %) | 11 (78,6 %) |
| смежные темы | 4 | 2 (50,0 %) | 2 (50,0 %) |
| вне темы | 14 | 14 (100,0 %) | 14 (100,0 %) |
| Всего | 48 | 46 (95,8 %) | 43 (89,6 %) |

По судье: ответил по существу вместо отказа - 2,1 %, вымышленный ответ - 12,5 %.


In [24]:
if RJ:
    sel = [r for r in RJ if r["judge"]["refusal_correct"] < 2 or r["judge"].get("hallucination") or r["qid"] in MANUAL_NOT_OK]
    md_table(["№", "Категория", "Судья", "Выдумка (судья)", "Ручная", "Вопрос", "Начало ответа"],
             [[r["qid"], CATN.get(r["category"], r["category"]), r["judge"]["refusal_correct"], "да" if r["judge"].get("hallucination") else "",
               "нет" if r["qid"] in MANUAL_NOT_OK else "да", RA.get(r["qid"], {}).get("query", ""),
               RA.get(r["qid"], {}).get("answer", "")[:200].replace("\n", " ").replace("|", "/") + "…"] for r in sel])

| № | Категория | Судья | Выдумка (судья) | Ручная | Вопрос | Начало ответа |
|---|---|---|---|---|---|---|
| R17 | новые технологии | 2 | да | нет | Какие требования предъявляются к зарядным станциям для электромобилей? | В предоставленном контексте нет специфических требований к зарядным станциям для электромобилей. Однако, можно сделать выводы на основе общих требований к зарядным устройствам и аккумуляторным батарея… |
| R22 | новые технологии | 2 | да | нет | Как прокладываются кабели витой пары категории 6 для сетей Ethernet? | В приведенном контексте нет информации о специфических требованиях к прокладке кабелей витой пары категории 6 для сетей Ethernet. Однако, согласно общим правилам, можно сделать следующие выводы:  1. В… |
| R26 | смежные темы | 1 | да | нет | Какие требования предъявляются к электрическому тёплому полу с нагревательным кабелем в ванной комнате? | Для электрического тёплого пола с нагревательным кабелем в ванной комнате следует учитывать следующие требования:  - Нагревательные элементы, замоноличенные в пол, должны быть покрыты заземленной мета… |
| R28 | новые технологии | 2 | да | нет | Как подключить к сети гибридный инвертор с аккумуляторами для резервного питания квартиры? | Вопрос не имеет прямого ответа в предоставленных фрагментах Правил устройства электроустановок (ПУЭ). Однако, можно сделать выводы на основе имеющейся информации:  1. Для обеспечения резервного питани… |
| R32 | смежные темы | 0 | да | нет | С какой периодичностью проводится тепловизионный контроль контактных соединений в распределительных устройствах? | Тепловизионный контроль контактных соединений, включая шины и контактные соединения, должен проводиться не реже одного раза в год, в период с июня по август, согласно пункту 63 ПУЭ.… |
| R47 | вне темы | 2 | да | да | Какие требования предъявляются к электрооборудованию парусных яхт? | Требования к электрооборудованию парусных яхт не приведены в предоставленном контексте Правил устройства электроустановок (ПУЭ). Однако, согласно пункту 1.7.155 ПУЭ, судовые электроустановки, включая … |

**Расхождения с судьёй.** В R17 (зарядные станции электромобилей), R22 (витая пара для Ethernet) и R28 (гибридный инвертор) модель сначала пишет, что в контексте ответа нет, а затем перечисляет пункты о зарядных устройствах аккумуляторов, контрольных кабелях и АВР так, будто это ответ на вопрос; судья ставит за это 2, при ручной проверке это 1.

**Явные ошибки.** R26 (тёплый пол в ванной): модель перечислила требования 7.1.88 к нагревательным элементам без оговорки, что специальных требований к тёплому полу в ПУЭ нет (судья: 1). R32 (периодичность тепловизионного контроля): модель ответила «не реже одного раза в год, пункт 63 ПУЭ» - это пункт приказа № 811 о другом, то есть ошибка источника и вымышленный ответ (судья: 0).

R47 (парусные яхты): судья отметил вымышленный ответ, но ответ корректен - модель отказалась и сослалась на 1.7.155, где сказано, что требования к передвижным электроустановкам не распространяются на судовые; при ручной проверке засчитан.

**Пример проверки правильного отказа (R17 и R32).** Ответ модели и оценка судьи берутся из журналов `answers.jsonl` и `judgments.jsonl` с меткой `refusal`, причина отсутствия ответа - из `eval_questions_refusal.json`, ручная проверка - список `MANUAL_NOT_OK`.

In [25]:
_rf = ROOT / "V2" / "data" / "rag_eval_questions" / "eval_questions_refusal.json"
_RQ = {}
if _rf.is_file():
    _q = json.loads(_rf.read_text(encoding="utf-8")); _RQ = {str(q["id"]): q for q in (_q["questions"] if isinstance(_q, dict) else _q)}
_RJB = {str(r["qid"]): r for r in RJ}
for q in ["R17", "R32"]:
    a, j = RA.get(q) if RJ else None, _RJB.get(q)
    if not a or not j:
        print(f"{q}: записи в журнале нет"); continue
    print(f"=== {q}. {a['query']}")
    print("Почему ответа нет в ПУЭ:", _RQ.get(q, {}).get("why_not_in_pue", "-"))
    print("Пункты поданных фрагментов:", ", ".join(c["clause"] + (" (приказ № 811)" if "811" in (c.get("document") or "") else "") for c in a["contexts"]))
    print(f"Время ответа: {num(a['elapsed_s'])} с")
    print("Ответ модели:\n" + a["answer"].strip())
    print(f"Судья: refusal_correct = {j['judge']['refusal_correct']}, вымысел - {'да' if j['judge'].get('hallucination') else 'нет'}; "
          f"ручная проверка: {'не засчитан' if q in MANUAL_NOT_OK else 'засчитан'}")
    print("Комментарий судьи:", j["judge"].get("comment", ""), "\n")

=== R17. Какие требования предъявляются к зарядным станциям для электромобилей?
Почему ответа нет в ПУЭ: Зарядные станции электромобилей в ПУЭ не рассматриваются; в 7.3.42 упоминаются только зарядные станции тяговых и стартерных аккумуляторных батарей.
Пункты поданных фрагментов: 4.4.4, 5.1.15, 4.4.5, 1.1.23, 5.1.1
Время ответа: 13,6 с
Ответ модели:
В предоставленном контексте нет специфических требований к зарядным станциям для электромобилей. Однако, можно сделать выводы на основе общих требований к зарядным устройствам и аккумуляторным батареям:

1. Зарядное устройство должно обеспечивать заряд аккумуляторной батареи на 90% номинальной емкости в течение не более 8 часов (ПУЭ, 4.4.4).
2. Аккумуляторные батареи могут устанавливаться в ЭМП при условии устройства вытяжного приспособления или зарядки в специальных помещениях или шкафах (ПУЭ, 5.1.15 п. 5).
3. Аккумуляторная установка должна быть оборудована вольтметром с переключателем и амперметрами (ПУЭ, 4.4.5).

Для зарядных станций дл

### 14.6. Ограничения оценки

Судья - та же модель, что отвечала, поэтому его оценки вспомогательные. Проверка на наборе без ответа показывает, что судья менее строг, чем человек: на 48 вопросах он засчитал 46 отказов, ручная проверка - 43. Ручная проверка ответов основного набора (выборка и все 19 критических) пока не выполнена; набор на отказ невелик (48 вопросов), смежных тем в нём только 4. Ответы оценены по одному прогону генерации.

### 14.7. Пример работы в режиме вопрос-ответ

Ниже - реальные записи из журналов прогона: что спросили, какие фрагменты подал поиск, что ответила модель и как ответ оценил судья. Примеры: верный ответ на исходный вопрос (1), верный ответ на часть вопроса с числами (49.6), ответ с неверно названным пунктом при поданном эталонном фрагменте (56.7), ошибка из-за поиска (164) и правильный отказ на вопрос без ответа в ПУЭ (R01).

Цепочка скриптов для одного вопроса:

1. `V2/scripts/run_pipeline.py search` (опыт 4: E5 с префиксами, 50 кандидатов, реранкер `BAAI/bge-reranker-v2-m3`) - выдача в `retrieval.json`;
2. `V2/scripts/generation/generate.py` - берёт из выдачи первые 5 фрагментов, формирует запрос с системной подсказкой PUE_SYSTEM из `src/rag/rag_service.py` и вызывает qwen2.5:14b-instruct-q4_K_M через Ollama; журнал - `answers.jsonl` (вопрос, поданные фрагменты, ответ, токены, время);
3. `V2/scripts/generation/judge.py` - оценка ответа по эталону; журнал - `judgments.jsonl` (баллы, отметки и комментарий судьи).

Для вопросов без ответа в ПУЭ поиск выполняет `V2/scripts/generation/refusal_search.py`, далее те же `generate.py` и `judge.py`.

In [26]:
QA_IDS = ["1", "49.6", "56.7", "164", "R01"]
ANS_BY = {str(r["qid"]): r for r in ANS}
JR_BY = {str(r["qid"]): r for r in JR}
RA_BY, RJ_BY = (RA if RJ else {}), {str(r["qid"]): r for r in RJ}
QR = {}
_rf = ROOT / "V2" / "data" / "rag_eval_questions" / "eval_questions_refusal.json"
if _rf.is_file():
    _q = json.loads(_rf.read_text(encoding="utf-8"))
    QR = {str(q["id"]): q for q in (_q["questions"] if isinstance(_q, dict) else _q)}
def show_qa(qid):
    a, j = (RA_BY.get(qid), RJ_BY.get(qid)) if qid.startswith("R") else (ANS_BY.get(qid), JR_BY.get(qid))
    if not a:
        print(f"Вопрос {qid}: записи в журнале нет"); return
    q = QR.get(qid, {}) if qid.startswith("R") else QS.get(qid, {})
    ref = "ответа в ПУЭ нет (" + CATN.get(q.get("category", ""), q.get("category", "")) + ")" if qid.startswith("R") else q.get("expected_clause", "-")
    kind = "вопрос без ответа в ПУЭ" if qid.startswith("R") else ("исходный вопрос" if a.get("is_original") else "часть вопроса")
    display(Markdown(f"#### Вопрос {qid} ({kind})\n\n**Вопрос:** {a['query']}  \n**Эталон:** {ref}"))
    md_table(["Место", "Пункт", "Документ", "Символов", "Обрезан"],
             [[c["rank"], c["clause"] or "-", Path(str(c["document"]).replace("\\", "/")).name, c["chars"], "да" if c["truncated"] else "нет"] for c in a["contexts"]])
    display(Markdown("**Ответ модели:**\n\n> " + a["answer"].strip().replace("\n", "\n> ")))
    print(f"Время ответа {num(a['elapsed_s'])} с; токенов в запросе {a['prompt_tokens']}, в ответе {a['answer_tokens']}"
          + ("; ответ достиг предела 512 токенов" if a.get("answer_hit_limit") else ""))
    if j:
        g = j["judge"]
        if qid.startswith("R"):
            print(f"Судья: правильный отказ {g.get('refusal_correct')} из 2; вымышленный ответ - {'да' if g.get('hallucination') else 'нет'}"
                  + (f"; ручная проверка: {'не засчитан' if qid in MANUAL_NOT_OK else 'засчитан'}"))
        else:
            print("Судья (0-2): " + ", ".join(f"{CS[c]} {g[c]}" for c, _ in CRIT)
                  + f"; критическая ошибка - {'да' if g.get('critical_error') else 'нет'}; эталонный пункт среди фрагментов - "
                  + ("да" if j["auto"]["required_clause_in_context"] else "нет"))
        if g.get("comment"):
            print("Комментарий судьи:", g["comment"])
for qid in QA_IDS:
    show_qa(qid)

#### Вопрос 1 (исходный вопрос)

**Вопрос:** Какое определение соответствует термину «электрическая сеть»?  
**Эталон:** 1.2.6

| Место | Пункт | Документ | Символов | Обрезан |
|---|---|---|---|---|
| 1 | 1.2.6 | 1.2 | 265 | нет |
| 2 | 1.2.3 | 1.2 | 135 | нет |
| 3 | 1.7.4 | 1.7 | 438 | нет |
| 4 | 1.2.2 | 1.2 | 312 | нет |
| 5 | 1.2.5 | 1.2 | 317 | нет |

**Ответ модели:**

> Термин «электрическая сеть» соответствует определению, данному в ПУЭ, глава 1.2, пункт 1.2.6:
> 
> "Электрическая сеть - совокупность электроустановок для передачи и распределения электрической энергии, состоящая из подстанций, распределительных устройств, токопроводов, воздушных и кабельных линий электропередачи, работающих на определенной территории."

Время ответа 15,4 с; токенов в запросе 755, в ответе 125
Судья (0-2): Факты 2, Числа 2, Исключения 2, Источник 2, Отказ 2; критическая ошибка - нет; эталонный пункт среди фрагментов - да
Комментарий судьи: Ответ модели полностью соответствует эталону и передает все обязательные факты без искажений.


#### Вопрос 49.6 (часть вопроса)

**Вопрос:** На какой высоте должны устанавливаться токопроводы и ограждения, размещаемые над проходами?  
**Эталон:** 2.2.20

| Место | Пункт | Документ | Символов | Обрезан |
|---|---|---|---|---|
| 1 | 2.2.20 | 2.2 | 1520 | нет |
| 2 | 7.10.27 | 7.10 | 1585 | нет |
| 3 | 4.1.23 | 4.1 | 2403 | нет |
| 4 | 2.2.27 | 2.2 | 1410 | нет |
| 5 | 2.2.28 | 2.2 | 631 | нет |

... (10 outputs left out)

#### Вопрос R01 (вопрос без ответа в ПУЭ)

**Вопрос:** Как рассчитывается оплата электроэнергии по двухставочному тарифу?  
**Эталон:** ответа в ПУЭ нет (другие нормы)

| Место | Пункт | Документ | Символов | Обрезан |
|---|---|---|---|---|
| 1 | 1.5.2 | 1.5 | 234 | нет |
| 2 | 2.4.11 | 2.4 | 1098 | нет |
| 3 | 1.5.11 | 1.5 | 2140 | нет |
| 4 | 1.5.12 | 1.5 | 1149 | нет |
| 5 | - | Приказ Минэнерго РФ от 12.08.2022 N 811 Об утверждении Правил технической эксплуатации | 1055 | нет |

**Ответ модели:**

> В предоставленном контексте нет информации о расчете оплаты электроэнергии по двухставочному тарифу. Пожалуйста, предоставьте дополнительную информацию или уточните вопрос, если требуется конкретная статья или пункт Правил устройства электроустановок (ПУЭ) на эту тему.

Время ответа 30,3 с; токенов в запросе 2421, в ответе 76
Судья: правильный отказ 2 из 2; вымышленный ответ - нет; ручная проверка: засчитан
Комментарий судьи: Модель правильно указала, что в предоставленных фрагментах ПУЭ нет информации по вопросу, но не предоставила дополнительную информацию или уточнение вопроса, как это могло бы быть полезно.


## 15. Ноутбук или .py файл (прототип)

### 15.1. Ноутбук

Ноутбук второй части `Etap4_V2.ipynb` содержит те же разделы 9-17 и те же номера рисунков и таблиц, что в отчёте. Он сам находит данные (папка `data_V2` рядом с ноутбуком или корень проекта), читает файлы метрик `metrics.json` из `V2/results/retrieval/split` и `split_extra`, определяет опыт по настройкам внутри файла и строит таблицы и графики разделов 13 и 14 (ответы и оценки судьи берутся из файлов `answers.jsonl` и `judgments.jsonl`). Milvus и Ollama для этого не нужны: достаточно выполнить все ячейки (Run All). Флаги в ячейке подготовки: `RUN_LIVE = True` заново выполняет поиск по готовым коллекциям Milvus, `RUN_EVAL = True` пересчитывает метрики из сохранённой выдачи `retrieval.json` модулем расчёта метрик `scoring.py` (оба флага работают только внутри репозитория). Описание всех файлов - в `Readme-4_V2.md`.

### 15.2. Скрипты и вызов модели

Модули конвейера второй версии перечислены в таблице 20 (раздел 12.2). Запускаются они через скрипты, приведённые в таблице 32.

**Таблица 32 - Скрипты второй версии (V2/scripts/)**

| Файл | Назначение |
|---|---|
| `V2/scripts/run_pipeline.py` | все этапы конвейера поиска: prepare-documents, load, load-clip, search, evaluate и all (все этапы подряд); вход - файл конфигурации из V2/configs, выход - retrieval.json и metrics.json |
| `V2/scripts/generation/generate.py` | ответы модели по сохранённой выдаче поиска; вход - retrieval.json и файл вопросов, выход - answers.jsonl и run.json (параметры прогона и контрольные суммы входных файлов) |
| `V2/scripts/generation/judge.py` | оценка ответов LLM-судьёй по пяти критериям; вход - answers.jsonl, выход - judgments.jsonl (оценка по каждому ответу) и judge_metrics.json (сводка) |
| `V2/scripts/generation/refusal_search.py` | поиск конфигурацией опыта 4 по 48 вопросам без ответа в ПУЭ (eval_questions_refusal.json); выход - retrieval.json в папке refusal_retrieval |
| `V2/scripts/collect_data_V2.py` | сборка папки data_V2 для этого ноутбука из результатов в репозитории (только копирование) |

Вызов модели: qwen2.5:14b-instruct-q4_K_M через сервис Ollama (http://localhost:11434); системная подсказка PUE_SYSTEM взята из первой версии (`src/rag/rag_service.py`) без изменений; модели подаются вопрос и первые 5 фрагментов выдачи. Остальные параметры - в таблице 28 (раздел 14.1).

### 15.3. Порядок запуска

Поиск и оценка выдачи (Windows, PowerShell, из корня проекта, Milvus запущен):

```
# опыт 1
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split_extra.json --mode base --allow-db-write
# опыт 2
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode base --allow-db-write
# контроль
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode e5 --allow-db-write
# опыт 3
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode rerank --allow-db-write
# опыт 4
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split_extra.json --mode rerank --reuse-collection
# опыт 5
python V2/scripts/run_pipeline.py load-clip --config V2/configs/pipeline_manual_merged_split.json --allow-db-write
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode rerank --reuse-collection --with-clip
```

Генерация, оценка судьёй и проверка правильного отказа (Ollama запущена) - команды `generate.py`, `judge.py` и `refusal_search.py` приведены в отчёте (раздел 15.3). Сборка папки данных для ноутбука: `python V2/scripts/collect_data_V2.py`.

In [27]:
import subprocess, sys
CFG = {"o1": ("pipeline_manual_merged_split_extra.json", "base"), "o2": ("pipeline_manual_merged_split.json", "base"),
       "o3": ("pipeline_manual_merged_split.json", "rerank"), "o4": ("pipeline_manual_merged_split_extra.json", "rerank"),
       "o5": ("pipeline_manual_merged_split.json", "rerank")}
def run(cmd):
    print(">", " ".join(cmd)); rc = subprocess.run(cmd, cwd=str(ROOT)).returncode
    if rc: raise RuntimeError(f"код {rc}")
if (RUN_LIVE or RUN_EVAL) and not (ROOT / "V2" / "scripts" / "run_pipeline.py").is_file():
    print("Данные читаются из data_V2: скриптов прогона здесь нет. Повторный расчёт выполняется внутри репозитория Diplom_AI.")
elif RUN_LIVE:
    for e in EXP:
        cfg, mode = CFG[e]
        run([sys.executable, "V2/scripts/run_pipeline.py", "all", "--config", f"V2/configs/{cfg}", "--mode", mode, "--reuse-collection"] + (["--with-clip"] if e == "o5" else []))
elif RUN_EVAL:
    for e in EXP:   # создаётся новая папка, старые файлы не меняются
        cfg, mode = CFG[e]
        run([sys.executable, "V2/scripts/run_pipeline.py", "evaluate", "--config", f"V2/configs/{cfg}", "--mode", mode,
             "--input", str((RUNS[e][0].parent / "retrieval.json").relative_to(ROOT))])
else:
    print("RUN_LIVE = False и RUN_EVAL = False - используются сохранённые metrics.json.")

RUN_LIVE = False и RUN_EVAL = False - используются сохранённые metrics.json.


### 15.4. Расположение результатов

Каждый прогон поиска пишет папку в `V2/results/retrieval/split` (опыты 2, 3, 5 и контроль) или `split_extra` (опыты 1 и 4) с двумя файлами: `retrieval.json` - выдача поиска (первые 20 фрагментов по каждому из 720 вопросов) и `metrics.json` - метрики. Ответы модели - в `V2/results/generation/answers/<папка>_rerank50/` (answers.jsonl, run.json), оценки судьи - в `V2/results/generation/judge/<папка>_rerank50/` (judgments.jsonl, judge_metrics.json); для проверки отказа - папки с окончанием `_refusal`. Журналы прогонов - `V2/results/logs/`. В пакет `data_V2` входят все эти файлы, кроме `retrieval.json` и журналов.


## 16. Выводы второй версии

- По сравнению с первым проектом поиск заметно улучшился. При том же правиле зачёта (по пункту), тех же моделях и тех же 268 вопросах Hit@1 опыта с реранкером вырос с 72,0 % до 87,7 %, Hit@5 - с 88,8 % до 95,5 %. Основная причина - исправленный корпус (раздел 13.6).
- Префиксы E5 дают небольшой, но устойчивый прирост. На 268 исходных вопросах Hit@1 растёт с 68,7 % до 71,6 % (на 8 вопросов больше), Hit@5 - с 88,8 % до 91,0 %; на всех 720 вопросах Hit@1 +4,3 п.п. По Hit@1 префиксы улучшили результат в 16 вопросах и ухудшили в 8, что заметно больше расхождения контрольного повтора.
- Основной прирост даёт реранкер. Переранжирование 20 кандидатов поднимает Hit@1 с 71,6 % до 88,1 % (на 44 вопроса больше), MRR - с 0,800 до 0,915. Результат улучшился в 52 вопросах и ухудшился в 8.
- Увеличение числа кандидатов с 20 до 50 повышает полноту поиска. Hit@5 растёт с 95,5 % до 98,5 %, «все факты в первых 5» - с 92,5 % до 96,3 %. При 20 кандидатах часть эталонных чанков не попадает в список, который видит реранкер (Hit@20 опыта 2 = 96,3 %). При этом растёт время обработки: переранжирование 50 кандидатов для 720 вопросов на RTX 5060 Ti заняло около 20 минут (по логу прогона); систематически скорость не измерялась.
- CLIP в текущем виде не повышает качество поиска. Опыт 5 отличается от опыта 4 на один вопрос в каждую сторону (улучшение по Hit@1 в одном вопросе, ухудшение по Hit@5 в одном), что меньше расхождения контрольного повтора. Видимый прирост CLIP относительно опыта 3 объясняется числом кандидатов, а не изображениями. При этом сами изображения в зачёт пока не входят, поэтому вывод относится только к влиянию CLIP на порядок текстовых чанков.
- Разбиение вопросов на части оправдано. Без реранкера части находятся реже исходных, с реранкером - на том же уровне или лучше. Это значит, что узкий вопрос по одному пункту система находит так же надёжно, как общий.
- Слабые места: раздел 1 (общие правила, в том числе 1.7 и 1.8 с рисунками и таблицами испытаний) и вопросы с общей формулировкой. Четыре вопроса из 268 не попадают в первые пять даже в лучшем опыте (опыт 4).
- Рекомендуемая конфигурация для бота: опыт 4 - E5 с префиксами, реранкер bge-reranker-v2-m3 по 50 кандидатам, без CLIP. Hit@1 89,9 %, Hit@5 98,5 %, MRR 0,937 на исходных вопросах.
- Генерация по выдаче опыта 4 работает на всех 720 вопросах без сбоев. Среднее время ответа модели - 5,2 с, номер пункта ПУЭ указан в 98,1 % ответов.
- Качество ответов по оценке судьи высокое. Итоговый балл 93,3 % (исходные 92,4 %, части 93,9 %), все пять критериев на максимуме у 85,1 % ответов, критические ошибки - в 19 ответах (2,6 %), из них почти половина связана с тем, что поиск не передал модели нужный пункт. Наименьший балл получен по критерию «исключения и условия» (90,8 %).
- Если ответа в ПУЭ нет, модель в большинстве случаев корректно отказывается от ответа. На 48 вопросах без ответа правильный отказ - 95,8 % по судье и 89,6 % по ручной проверке. Слабое место - смежные темы: модель приводит близкие по теме пункты в качестве ответа.
- Ограничения этих выводов: один прогон на опыт (кроме контрольного повтора опыта 2), одна текстовая модель, качество ответов оценено судьёй - той же моделью, что отвечала (ручная проверка выполнена только для набора на отказ), изображения не оцениваются.

## 17. План работы по итогам второй версии

- Генерация и оценка судьёй выполнены (разделы 14.2-14.5). Осталось: ручная проверка выборки ответов основного набора и всех 19 критических ошибок, чтобы оценить согласие судьи с человеком; при возможности - судья другой моделью.
- Набор вопросов без ответа в ПУЭ составлен и проверен (раздел 14.5). Осталось: расширить категорию смежных тем (сейчас 4 вопроса) и проверить, снижает ли уточнение системной подсказки подстановку посторонних пунктов.
- Оценка выдачи изображений: включить в зачёт факты с рисунками и таблицами-изображениями и только после этого делать вывод о пользе CLIP.
- Замер скорости: время поиска, переранжирования и генерации на вопрос. Время генерации измерено (раздел 14.2); время поиска и переранжирования пока не замерено.
- Подключение рекомендованной конфигурации к Telegram-боту.
- Версия V3: универсальный конвейер добавления и обновления документов корпуса.

## Приложения ко второй версии

А. Файлы результатов. Каждый прогон записал папку с retrieval.json (выдача, 62-68 МБ) и metrics.json (метрики по каждому вопросу и сводка) в V2/results/retrieval/split (опыты 2, 3, 5, контроль) и V2/results/retrieval/split_extra (опыты 1, 4). Журналы прогонов - в V2/results/logs/ .

Б. Порядок прогонов: опыт 2, контроль, опыт 3, опыт 5, опыт 1, опыт 4.